# Moons, Stars, and Clever Hans: What Did the Model Actually Learn?

**Accuracy says both models learned the task. Moving the same moon shows one of them learned the wrong thing.**

This notebook is a controlled Clever-Hans laboratory. Ordinary validation asks whether a model is right on a familiar exam. Behavioural XAI asks whether it is right for the right reason.


## 0. Setup and deterministic generator

We keep the experiment deterministic and generated in-notebook. Static plots and animations are displayed inline only. The notebook does not save figure or animation artefacts.

**Factorised generator.** We view each image as

\[
X = R(Y, A, N, \varepsilon),
\]

where \(Y\) is the semantic label, \(A\) contains semantic shape factors, \(N\) contains nuisance factors such as position or background, \(R\) is the renderer, and \(\varepsilon\) is noise. The intended model should respond to semantic shape, not nuisance factors.

The visible task is moon versus star. Act I hides an absolute-position shortcut: moons usually appear lower-left and stars upper-right. We do not reveal that to the audience yet.

In [ ]:
# ruff: noqa
from __future__ import annotations

import math
import base64
import random
from dataclasses import dataclass
from pathlib import Path
from io import BytesIO
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, Markdown, display
from PIL import Image, ImageDraw, ImageFilter, ImageFont

SEED = 1701
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)

plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.labelsize": 12,
        "figure.dpi": 140,
        "savefig.dpi": 180,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def find_project_root() -> Path:
    """Return the repository root without assuming the notebook working directory."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the XAI project root.")


def find_notebook_path() -> Path:
    """Locate this notebook from either repo root or notebooks/shortcut_lab."""
    candidates = [
        Path.cwd() / "00_moons_stars_clever_hans.ipynb",
        Path.cwd() / "notebooks/shortcut_lab/00_moons_stars_clever_hans.ipynb",
        find_project_root() / "notebooks/shortcut_lab/00_moons_stars_clever_hans.ipynb",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate 00_moons_stars_clever_hans.ipynb")


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = find_notebook_path()
DEMO = "00 Moons/Stars Clever-Hans position shortcut"
DATA_MODE = "generated_controlled_demo"
EXTERNAL_DATA_REQUIRED = False
MANIFEST_PATH = "not applicable"
MANIFEST_EXISTS = False
DATASET_SOURCE = "Generated inside this notebook"
LICENCE_NOTE = "Repo-authored controlled generated data; no external dataset licence."
MISSING_FILES: list[str] = []
TRUE_TASK = "shape: moon versus star"
SHORTCUT = "absolute object position"
TRAINING_BIAS = "moons lower-left; stars upper-right"
BASELINE_MODEL = "Pixel MLP"
ROBUST_MODEL = "translation-aware CNN"
XAI_PROBES = "movement counterfactuals, response maps, shortcut statistics, confidence paths, saliency caution, re-test"

status_lines = [
    "DEMO: 00 Moons, Stars, and Clever Hans",
    f"DATA_MODE: {DATA_MODE}",
    f"EXTERNAL_DATA_REQUIRED: {str(EXTERNAL_DATA_REQUIRED).lower()}",
    "TRUE TASK: shape classification",
    "AUDIT QUESTION: did the model learn the shape, or a hidden rule?",
    "BASELINE MODEL: pixel MLP",
    "ROBUST MODEL: translation-aware CNN",
    "XAI PROBES: movement counterfactuals, confidence paths, response maps, shortcut statistics, saliency caution, re-test",
    f"MANIFEST_PATH: {MANIFEST_PATH}",
    f"MANIFEST_EXISTS: {MANIFEST_EXISTS}",
    f"PROJECT_ROOT: {PROJECT_ROOT}",
    f"DATASET_SOURCE: {DATASET_SOURCE}",
    f"LICENCE_NOTE: {LICENCE_NOTE}",
    f"MISSING_FILES: {MISSING_FILES}",
    f"SEED: {SEED}",
]
display(Markdown("```text\n" + "\n".join(status_lines) + "\n```"))

# Deterministic renderer and generated datasets.
IMAGE_SIZE = 128
MODEL_SIZE = 40
RENDER_SCALE = 3
CLASS_NAMES = {0: "moon", 1: "star"}
CLASS_COLOURS = {0: "#4C78A8", 1: "#F2A541"}
MODEL_COLOURS = {"MLP": "#C44E52", "CNN": "#2E8B57"}
LOWER_LEFT_REGION = (24, 48, 82, 108)  # x_min, x_max, y_min, y_max
UPPER_RIGHT_REGION = (82, 108, 18, 44)
FREE_REGION = (24, 108, 18, 108)
CANONICAL_CENTRES = {"lower_left": (36, 96), "upper_right": (96, 32)}
COMMON_GROUPS = {"moon_lower_left", "star_upper_right"}
CROSSED_GROUPS = {"moon_upper_right", "star_lower_left"}


@dataclass(frozen=True)
class ShapeSpec:
    label: int
    centre: tuple[int, int]
    seed: int
    size: float
    rotation: float
    brightness: int
    stroke: int
    crescent: float


@dataclass(frozen=True)
class PositionSample:
    image: Image.Image
    label: int
    position: str
    centre: tuple[int, int]
    group: str
    split: str
    object_seed: int
    spec: ShapeSpec


def rng_for(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


def position_name(centre: tuple[int, int]) -> str:
    x, y = centre
    if x < IMAGE_SIZE // 2 and y >= IMAGE_SIZE // 2:
        return "lower_left"
    if x >= IMAGE_SIZE // 2 and y < IMAGE_SIZE // 2:
        return "upper_right"
    return "uniform"


def group_name(label: int, position: str) -> str:
    return f"{CLASS_NAMES[label]}_{position}"


def choose_centre(region: tuple[int, int, int, int], rng: np.random.Generator) -> tuple[int, int]:
    return (
        int(rng.integers(region[0], region[1] + 1)),
        int(rng.integers(region[2], region[3] + 1)),
    )


def make_shape_spec(
    label: int,
    region: tuple[int, int, int, int],
    seed: int,
    fixed_centre: tuple[int, int] | None = None,
) -> ShapeSpec:
    local_rng = rng_for(seed)
    centre = fixed_centre if fixed_centre is not None else choose_centre(region, local_rng)
    return ShapeSpec(
        label=label,
        centre=centre,
        seed=seed,
        size=float(local_rng.uniform(20.0, 24.0)),
        rotation=float(local_rng.uniform(-16.0, 16.0)),
        brightness=int(local_rng.integers(228, 256)),
        stroke=int(local_rng.integers(1, 2)),
        crescent=float(local_rng.uniform(0.60, 0.70)),
    )


def move_spec(spec: ShapeSpec, centre: tuple[int, int]) -> ShapeSpec:
    return ShapeSpec(
        label=spec.label,
        centre=centre,
        seed=spec.seed,
        size=spec.size,
        rotation=spec.rotation,
        brightness=spec.brightness,
        stroke=spec.stroke,
        crescent=spec.crescent,
    )


def star_points(cx: float, cy: float, outer: float, inner: float, rotation: float) -> list[tuple[float, float]]:
    points = []
    for index in range(10):
        radius = outer if index % 2 == 0 else inner
        angle = math.radians(rotation - 90.0 + index * 36.0)
        points.append((cx + math.cos(angle) * radius, cy + math.sin(angle) * radius))
    return points


def object_mask(spec: ShapeSpec) -> Image.Image:
    canvas_size = IMAGE_SIZE * RENDER_SCALE
    mask = Image.new("L", (canvas_size, canvas_size), 0)
    draw = ImageDraw.Draw(mask)
    cx = spec.centre[0] * RENDER_SCALE
    cy = spec.centre[1] * RENDER_SCALE
    size = spec.size * RENDER_SCALE
    if spec.label == 1:
        draw.polygon(star_points(cx, cy, size, size * 0.45, spec.rotation), fill=255)
    else:
        draw.ellipse((cx - size, cy - size, cx + size, cy + size), fill=255)
        offset = size * spec.crescent
        draw.ellipse((cx - size + offset, cy - size * 1.06, cx + size + offset, cy + size * 1.06), fill=0)
    return mask.filter(ImageFilter.GaussianBlur(0.6 * RENDER_SCALE)).resize(
        (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS
    )


def render_shape(spec: ShapeSpec) -> Image.Image:
    local_rng = rng_for(spec.seed + 99)
    yy, xx = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE]
    texture = (
        38.0
        + 1.8 * np.sin(xx / 11.0)
        + 1.3 * np.cos(yy / 13.0)
        + local_rng.normal(0.0, 1.3, (IMAGE_SIZE, IMAGE_SIZE))
    )
    background = Image.fromarray(np.clip(texture, 0, 255).astype(np.uint8), "L")
    object_layer = Image.fromarray(np.full((IMAGE_SIZE, IMAGE_SIZE), spec.brightness, dtype=np.uint8), "L")
    return Image.composite(object_layer, background, object_mask(spec))


def make_sample(
    label: int,
    region: tuple[int, int, int, int],
    split: str,
    seed: int,
    fixed_centre: tuple[int, int] | None = None,
) -> PositionSample:
    spec = make_shape_spec(label, region, seed, fixed_centre=fixed_centre)
    position = position_name(spec.centre)
    return PositionSample(
        image=render_shape(spec),
        label=label,
        position=position,
        centre=spec.centre,
        group=group_name(label, position),
        split=split,
        object_seed=seed,
        spec=spec,
    )


def make_dataset(
    split: str,
    common_per_class: int,
    crossed_per_class: int,
    seed_offset: int,
    free_positions: bool = False,
) -> list[PositionSample]:
    samples: list[PositionSample] = []
    counter = seed_offset
    for label in (0, 1):
        common_region = LOWER_LEFT_REGION if label == 0 else UPPER_RIGHT_REGION
        crossed_region = UPPER_RIGHT_REGION if label == 0 else LOWER_LEFT_REGION
        for _ in range(common_per_class):
            region = FREE_REGION if free_positions else common_region
            samples.append(make_sample(label, region, split, SEED + counter))
            counter += 1
        for _ in range(crossed_per_class):
            region = FREE_REGION if free_positions else crossed_region
            samples.append(make_sample(label, region, split, SEED + counter))
            counter += 1
    random.Random(SEED + seed_offset).shuffle(samples)
    return samples


biased_train_samples = make_dataset("biased train", 600, 0, seed_offset=0)
iid_validation_samples = make_dataset("IID validation", 200, 0, seed_offset=10_000)
ood_audit_samples = make_dataset("balanced OOD audit", 60, 60, seed_offset=20_000)
position_augmented_train_samples = make_dataset(
    "position-augmented train", 300, 300, seed_offset=30_000, free_positions=True
)
position_augmented_validation_samples = make_dataset(
    "position-augmented validation", 120, 120, seed_offset=40_000, free_positions=True
)
uniform_audit_samples = make_dataset("uniform-position audit", 120, 0, seed_offset=50_000, free_positions=True)

print(f"Biased training samples: {len(biased_train_samples)}")
print(f"IID validation samples: {len(iid_validation_samples)}")
print(f"Balanced OOD audit samples: {len(ood_audit_samples)}")
print(f"position-augmented train samples: {len(position_augmented_train_samples)}")
print(f"Uniform-position audit samples: {len(uniform_audit_samples)}")


## 1. The apparent task: moon versus star

At first glance this looks like a normal shape-recognition task. The point of the demo is that this innocent view is incomplete.


In [ ]:
def show_fig(fig: plt.Figure) -> None:
    """Display a Matplotlib figure inline and close it. Static plots are not saved."""
    display(fig)
    plt.close(fig)


def show_pil_image(image: Image.Image) -> None:
    """Display a PIL image inline without writing a static PNG file."""
    display(image)


def add_panel_label(axis: plt.Axes, label: str) -> None:
    axis.text(
        0.02,
        0.98,
        label,
        transform=axis.transAxes,
        ha="left",
        va="top",
        fontsize=12,
        fontweight="bold",
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "#D0D0D0", "alpha": 0.95},
    )


def style_axes(axis: plt.Axes) -> None:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.grid(alpha=0.16)


def draw_confidence_bar(axis: plt.Axes, p_star: float, title: str, colour: str) -> None:
    axis.barh([0], [p_star], color=colour, height=0.42)
    axis.axvline(0.5, color="#333333", linestyle="--", linewidth=1.0)
    axis.set_xlim(0, 1)
    axis.set_yticks([])
    axis.set_title(title, fontsize=10.5, fontweight="bold")
    axis.text(min(p_star + 0.03, 0.92), 0, f"{p_star:.2f}", va="center", fontsize=10.5, fontweight="bold")


def image_array(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("L"), dtype=np.float32) / 255.0


def representative_sample(label: int, position: str) -> PositionSample:
    centre = CANONICAL_CENTRES[position]
    region = LOWER_LEFT_REGION if position == "lower_left" else UPPER_RIGHT_REGION
    return make_sample(label, region, "representative", SEED + 70_000 + label * 100, fixed_centre=centre)


def sample_at_centre(label: int, centre: tuple[int, int], seed: int, split: str = "presentation") -> PositionSample:
    region = (centre[0], centre[0], centre[1], centre[1])
    return make_sample(label, region, split, seed, fixed_centre=centre)


# Innocent-looking task figure. This deliberately avoids showing the training-position distribution.
apparent_examples = [
    sample_at_centre(0, (34, 36), SEED + 71_001),
    sample_at_centre(0, (64, 72), SEED + 71_002),
    sample_at_centre(0, (92, 42), SEED + 71_003),
    sample_at_centre(1, (38, 88), SEED + 71_101),
    sample_at_centre(1, (68, 34), SEED + 71_102),
    sample_at_centre(1, (96, 86), SEED + 71_103),
]
fig = plt.figure(figsize=(11.0, 6.2), constrained_layout=True)
grid = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 1.15])
fig.suptitle("The task humans think we gave the model", fontsize=19, fontweight="bold")
fig.text(0.5, 0.925, "Classify shape: moon or star.", ha="center", fontsize=13.5, color="#555555")
for idx, sample in enumerate(apparent_examples[:3]):
    axis = fig.add_subplot(grid[0, idx])
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_title("moon example", color=CLASS_COLOURS[0], fontweight="bold")
    axis.set_xticks([])
    axis.set_yticks([])
for idx, sample in enumerate(apparent_examples[3:]):
    axis = fig.add_subplot(grid[1, idx])
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_title("star example", color=CLASS_COLOURS[1], fontweight="bold")
    axis.set_xticks([])
    axis.set_yticks([])
audit_axis = fig.add_subplot(grid[:, 3])
audit_axis.axis("off")
audit_axis.add_patch(plt.Rectangle((0.04, 0.08), 0.92, 0.84, facecolor="#F8F8F8", edgecolor="#D0D0D0", linewidth=1.4, transform=audit_axis.transAxes))
audit_axis.text(0.11, 0.82, "Balanced-looking audit", fontsize=13.5, fontweight="bold", transform=audit_axis.transAxes)
audit_axis.text(0.11, 0.64, "moons and stars\nappear visually clear", fontsize=12, transform=audit_axis.transAxes)
audit_axis.text(0.11, 0.42, "The apparent question:\ncan the model recognise\nthe shape?", fontsize=12, transform=audit_axis.transAxes)
audit_axis.text(0.11, 0.18, "No external data.\nGenerated controlled demo.", fontsize=10.5, color="#666666", transform=audit_axis.transAxes)
fig.text(0.5, -0.02, "At this point the task looks like ordinary shape recognition.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)


## 2. Both models appear to work

We train a Pixel MLP and a CNN with position augmentation. Ordinary IID validation repeats the training bias, so both models appear to solve the task.

**Empirical risk minimisation.** Training approximately solves

\[
\hat f = \arg\min_f \frac{1}{n}\sum_i \ell(f(x_i), y_i).
\]

ERM does not know which predictive factor is meaningful. If a nuisance factor predicts the label, ERM is rewarded for using it.

In [ ]:
TENSOR_CACHE: dict[tuple[int, ...], tuple[torch.Tensor, torch.Tensor]] = {}
ACT1_TARGET_MARGIN = 6.0
ACT1_CONFIDENCE_MEAN_THRESHOLD = 0.98
ACT1_CONFIDENCE_MIN_THRESHOLD = 0.90


def samples_to_tensors(samples: list[PositionSample]) -> tuple[torch.Tensor, torch.Tensor]:
    cache_key = tuple(id(sample) for sample in samples)
    if cache_key in TENSOR_CACHE:
        return TENSOR_CACHE[cache_key]
    arrays = [
        np.asarray(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
        for sample in samples
    ]
    x = torch.tensor(np.stack(arrays)[:, None, :, :], dtype=torch.float32)
    y = torch.tensor([sample.label for sample in samples], dtype=torch.long)
    TENSOR_CACHE[cache_key] = (x, y)
    return x, y


class PixelMLP(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(MODEL_SIZE * MODEL_SIZE, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GapCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 24, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(24, 48, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(48, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d(1),
        )
        self.head = torch.nn.Linear(64, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x).flatten(1))


def binary_correct_class_probabilities(probabilities: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    row_ids = torch.arange(labels.shape[0], device=labels.device)
    return probabilities[row_ids, labels]


def act1_margin_loss(logits: torch.Tensor, y: torch.Tensor, target_margin: float = ACT1_TARGET_MARGIN) -> torch.Tensor:
    correct = logits.gather(1, y[:, None]).squeeze(1)
    wrong = logits.gather(1, (1 - y)[:, None]).squeeze(1)
    return torch.relu(target_margin - (correct - wrong)).mean()


def evaluate_model(model: torch.nn.Module, samples: list[PositionSample]) -> dict[str, Any]:
    x, y = samples_to_tensors(samples)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(logits, y).item()
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)
        correct_probs = binary_correct_class_probabilities(probabilities, y)
    return {
        "loss": loss,
        "accuracy": float((predictions == y).float().mean().item()),
        "mean_correct_prob": float(correct_probs.mean().item()),
        "min_correct_prob": float(correct_probs.min().item()),
        "star_score": probabilities[:, 1].cpu().numpy(),
        "correct_prob": correct_probs.cpu().numpy(),
        "predictions": predictions.cpu().numpy(),
        "labels": y.cpu().numpy(),
    }


def act1_confidence_ready(train_eval: dict[str, Any], iid_eval: dict[str, Any]) -> bool:
    return (
        train_eval["accuracy"] >= 0.995
        and iid_eval["accuracy"] >= 0.995
        and iid_eval["mean_correct_prob"] >= ACT1_CONFIDENCE_MEAN_THRESHOLD
        and iid_eval["min_correct_prob"] >= ACT1_CONFIDENCE_MIN_THRESHOLD
    )


def train_model(
    model: torch.nn.Module,
    train_samples: list[PositionSample],
    iid_samples: list[PositionSample],
    ood_samples: list[PositionSample],
    epochs: int,
    learning_rate: float,
) -> list[dict[str, float]]:
    x_train, y_train = samples_to_tensors(train_samples)
    optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=5e-5)
    history: list[dict[str, float]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimiser.zero_grad()
        logits = model(x_train)
        loss = torch.nn.functional.cross_entropy(logits, y_train) + 0.10 * act1_margin_loss(logits, y_train)
        loss.backward()
        optimiser.step()
        train_eval = evaluate_model(model, train_samples)
        iid_eval = evaluate_model(model, iid_samples)
        ood_eval = evaluate_model(model, ood_samples)
        history.append(
            {
                "epoch": float(epoch),
                "train_loss": float(train_eval["loss"]),
                "train_accuracy": float(train_eval["accuracy"]),
                "train_mean_correct_prob": float(train_eval["mean_correct_prob"]),
                "train_min_correct_prob": float(train_eval["min_correct_prob"]),
                "iid_loss": float(iid_eval["loss"]),
                "iid_accuracy": float(iid_eval["accuracy"]),
                "iid_mean_correct_prob": float(iid_eval["mean_correct_prob"]),
                "iid_min_correct_prob": float(iid_eval["min_correct_prob"]),
                "ood_loss": float(ood_eval["loss"]),
                "ood_accuracy": float(ood_eval["accuracy"]),
            }
        )
        if epoch >= 18 and act1_confidence_ready(train_eval, iid_eval):
            break
    return history


baseline_mlp = PixelMLP()
mitigated_cnn = GapCNN()
mlp_history = train_model(
    baseline_mlp,
    biased_train_samples,
    iid_validation_samples,
    ood_audit_samples,
    epochs=80,
    learning_rate=2.5e-3,
)
cnn_history = train_model(
    mitigated_cnn,
    position_augmented_train_samples,
    position_augmented_validation_samples,
    ood_audit_samples,
    epochs=120,
    learning_rate=3.0e-3,
)


def ensure_iid_convergence() -> None:
    global baseline_mlp, mitigated_cnn, mlp_history, cnn_history
    mlp_train = evaluate_model(baseline_mlp, biased_train_samples)
    mlp_iid = evaluate_model(baseline_mlp, iid_validation_samples)
    cnn_train = evaluate_model(mitigated_cnn, position_augmented_train_samples)
    cnn_iid = evaluate_model(mitigated_cnn, position_augmented_validation_samples)
    if not act1_confidence_ready(mlp_train, mlp_iid):
        torch.manual_seed(SEED + 101)
        baseline_mlp = PixelMLP()
        mlp_history = train_model(
            baseline_mlp,
            biased_train_samples,
            iid_validation_samples,
            ood_audit_samples,
            epochs=120,
            learning_rate=2.8e-3,
        )
    if not act1_confidence_ready(cnn_train, cnn_iid):
        torch.manual_seed(SEED + 102)
        mitigated_cnn = GapCNN()
        cnn_history = train_model(
            mitigated_cnn,
            position_augmented_train_samples,
            position_augmented_validation_samples,
            ood_audit_samples,
            epochs=160,
            learning_rate=3.2e-3,
        )
    mlp_iid_final = evaluate_model(baseline_mlp, iid_validation_samples)
    cnn_iid_final = evaluate_model(mitigated_cnn, position_augmented_validation_samples)
    if not act1_confidence_ready(evaluate_model(baseline_mlp, biased_train_samples), mlp_iid_final):
        raise RuntimeError("Act I failed: Pixel MLP did not reach decisive IID confidence.")
    if not act1_confidence_ready(evaluate_model(mitigated_cnn, position_augmented_train_samples), cnn_iid_final):
        raise RuntimeError("Act I failed: CNN did not reach decisive IID confidence.")


ensure_iid_convergence()
print("Trained MLP on biased positions with decisive IID confidence.")
print("Trained CNN with shared convolution, global average pooling, position augmentation, and decisive IID confidence.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.8), constrained_layout=True)
fig.suptitle("Everything looks fine — if the test repeats the bias", fontsize=18, fontweight="bold")

card_data = [
    ("Pixel MLP", mlp_history[-1]["train_accuracy"], mlp_history[-1]["iid_accuracy"], mlp_history[-1]["iid_mean_correct_prob"], mlp_history[-1]["iid_min_correct_prob"], MODEL_COLOURS["MLP"]),
    ("CNN + augmentation", cnn_history[-1]["train_accuracy"], cnn_history[-1]["iid_accuracy"], cnn_history[-1]["iid_mean_correct_prob"], cnn_history[-1]["iid_min_correct_prob"], MODEL_COLOURS["CNN"]),
]
for axis, (model_name, train_acc, iid_acc, iid_mean_prob, iid_min_prob, colour) in zip(axes[:2], card_data, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.04, 0.08), 0.92, 0.82, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.4, transform=axis.transAxes))
    axis.text(0.10, 0.75, model_name, fontsize=15, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.10, 0.52, f"train accuracy\n{train_acc * 100:.1f}%", fontsize=14, transform=axis.transAxes)
    axis.text(0.10, 0.30, f"IID validation\n{iid_acc * 100:.1f}%", fontsize=14, fontweight="bold", transform=axis.transAxes)
    axis.text(0.10, 0.15, f"mean correct probability {iid_mean_prob:.2f}\nminimum correct probability {iid_min_prob:.2f}", fontsize=10.5, color="#555555", transform=axis.transAxes)

curve_axis = axes[2]
mlp_epochs = [row["epoch"] for row in mlp_history]
cnn_epochs = [row["epoch"] for row in cnn_history]
curve_axis.plot(mlp_epochs, [row["iid_accuracy"] for row in mlp_history], color=MODEL_COLOURS["MLP"], linewidth=2.6, label="MLP IID")
curve_axis.plot(cnn_epochs, [row["iid_accuracy"] for row in cnn_history], color=MODEL_COLOURS["CNN"], linewidth=2.6, label="CNN IID")
curve_axis.set_title("ordinary validation")
curve_axis.set_xlabel("epoch")
curve_axis.set_ylabel("accuracy")
curve_axis.set_ylim(0.0, 1.05)
curve_axis.grid(alpha=0.18)
curve_axis.legend(frameon=False, loc="lower right")
fig.text(0.5, -0.02, "At this point a normal validation report would say both models learned the task.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)


## Many functions pass the same exam

> **The training data does not identify the human concept. Many different functions solve the biased exam perfectly. XAI reveals which function the model actually learned.**

\[
\hat f = \arg\min_f \frac{1}{n}\sum_i \ell(f(x_i), y_i)
\]

ERM does not know which predictive factor is meaningful. If shape, position, and background all predict the label, all are valid low-loss solutions. The optimiser has no obligation to choose the human one.


In [ ]:

def quick_position_rule_accuracy(samples: list[PositionSample]) -> float:
    labels = np.array([sample.label for sample in samples], dtype=np.int64)
    predictions = np.array([sample.centre[0] - sample.centre[1] > 0 for sample in samples], dtype=np.int64)
    return float(np.mean(predictions == labels))


many_functions_position_iid_accuracy = quick_position_rule_accuracy(iid_validation_samples)
many_functions_position_audit_accuracy = quick_position_rule_accuracy(ood_audit_samples)

many_functions_rows = [
    {
        "rule": "Human rule",
        "evidence": "shape",
        "iid": "near 100%",
        "audit": "passes movement and swap probes",
        "intervention": "should stay stable",
        "colour": MODEL_COLOURS["CNN"],
    },
    {
        "rule": "Position shortcut",
        "evidence": "object address",
        "iid": f"{many_functions_position_iid_accuracy * 100:.1f}%",
        "audit": f"{many_functions_position_audit_accuracy * 100:.1f}% on balanced audit",
        "intervention": "fails when same object moves",
        "colour": MODEL_COLOURS["MLP"],
    },
    {
        "rule": "Background shortcut",
        "evidence": "invisible tint",
        "iid": "can be 100%",
        "audit": "fails under background swap",
        "intervention": "fails when cue is randomised",
        "colour": "#D89C21",
    },
    {
        "rule": "Mixed rule",
        "evidence": "shape + shortcuts",
        "iid": "can be 100%",
        "audit": "only factor probes separate it",
        "intervention": "changes when nuisance reward changes",
        "colour": "#6A4C93",
    },
]

fig, axis = plt.subplots(figsize=(14.6, 5.2), constrained_layout=True)
axis.axis("off")
fig.suptitle("100% IID accuracy does not tell us which rule was learned", fontsize=18.5, fontweight="bold")
columns = ["rule", "evidence used", "biased IID accuracy", "shortcut-breaking audit", "under intervention"]
col_x = [0.03, 0.22, 0.41, 0.60, 0.80]
for x, column in zip(col_x, columns, strict=True):
    axis.text(x, 0.90, column, fontsize=10.8, fontweight="bold", ha="left", transform=axis.transAxes)
for row, item in enumerate(many_functions_rows):
    y = 0.74 - row * 0.18
    axis.add_patch(plt.Rectangle((0.015, y - 0.062), 0.97, 0.118, facecolor="#FAFAFA", edgecolor="#E0E0E0", linewidth=1.0, transform=axis.transAxes))
    values = [item["rule"], item["evidence"], item["iid"], item["audit"], item["intervention"]]
    for x, value, column in zip(col_x, values, columns, strict=True):
        colour = item["colour"] if column in {"rule", "evidence used"} else "#222222"
        axis.text(x, y, value, fontsize=10.2, color=colour, fontweight="bold" if column == "rule" else "normal", ha="left", va="center", wrap=True, transform=axis.transAxes)
axis.text(0.5, 0.04, "Accuracy is not behaviour. Behavioural and statistical probes identify the controlling factor.", ha="center", fontsize=12.2, fontweight="bold", transform=axis.transAxes)
show_fig(fig)


## 3. Act I reveal: same object, different position

> **Same moon. Same pixels. Different place. Different answer.**

Now we hold the object fixed and move only its address on the canvas.


In [ ]:
def group_accuracies(model: torch.nn.Module, samples: list[PositionSample]) -> dict[str, float]:
    evaluation = evaluate_model(model, samples)
    predictions = evaluation["predictions"]
    labels = evaluation["labels"]
    groups = np.array([sample.group for sample in samples])
    return {
        group: float((predictions[groups == group] == labels[groups == group]).mean())
        for group in sorted(set(groups))
    }


def subset_accuracy(model: torch.nn.Module, samples: list[PositionSample], groups: set[str]) -> float:
    selected = [sample for sample in samples if sample.group in groups]
    return float(evaluate_model(model, selected)["accuracy"])


mlp_train_eval = evaluate_model(baseline_mlp, biased_train_samples)
mlp_iid_validation_eval = evaluate_model(baseline_mlp, iid_validation_samples)
mlp_ood_eval_for_metrics = evaluate_model(baseline_mlp, ood_audit_samples)
mlp_train_accuracy = float(mlp_train_eval["accuracy"])
mlp_iid_validation_accuracy = float(mlp_iid_validation_eval["accuracy"])
mlp_iid_validation_mean_correct_prob = float(mlp_iid_validation_eval["mean_correct_prob"])
mlp_iid_validation_min_correct_prob = float(mlp_iid_validation_eval["min_correct_prob"])
mlp_ood_accuracy = float(mlp_ood_eval_for_metrics["accuracy"])
mlp_common_accuracy = subset_accuracy(baseline_mlp, ood_audit_samples, COMMON_GROUPS)
mlp_crossed_accuracy = subset_accuracy(baseline_mlp, ood_audit_samples, CROSSED_GROUPS)
mlp_uniform_accuracy = float(evaluate_model(baseline_mlp, uniform_audit_samples)["accuracy"])
mlp_group_accuracies = group_accuracies(baseline_mlp, ood_audit_samples)
mlp_worst_group_accuracy = min(mlp_group_accuracies.values())

cnn_train_eval = evaluate_model(mitigated_cnn, position_augmented_train_samples)
cnn_iid_validation_eval = evaluate_model(mitigated_cnn, position_augmented_validation_samples)
cnn_ood_eval_for_metrics = evaluate_model(mitigated_cnn, ood_audit_samples)
cnn_train_accuracy = float(cnn_train_eval["accuracy"])
cnn_iid_validation_accuracy = float(cnn_iid_validation_eval["accuracy"])
cnn_iid_validation_mean_correct_prob = float(cnn_iid_validation_eval["mean_correct_prob"])
cnn_iid_validation_min_correct_prob = float(cnn_iid_validation_eval["min_correct_prob"])
cnn_ood_accuracy = float(cnn_ood_eval_for_metrics["accuracy"])
cnn_common_accuracy = subset_accuracy(mitigated_cnn, ood_audit_samples, COMMON_GROUPS)
cnn_crossed_accuracy = subset_accuracy(mitigated_cnn, ood_audit_samples, CROSSED_GROUPS)
cnn_uniform_accuracy = float(evaluate_model(mitigated_cnn, uniform_audit_samples)["accuracy"])
cnn_group_accuracies = group_accuracies(mitigated_cnn, ood_audit_samples)
cnn_worst_group_accuracy = min(cnn_group_accuracies.values())


def fixed_pair(label: int, usual_position: str, moved_position: str, seed: int) -> tuple[PositionSample, PositionSample]:
    usual_region = LOWER_LEFT_REGION if usual_position == "lower_left" else UPPER_RIGHT_REGION
    usual_spec = make_shape_spec(label, usual_region, seed, fixed_centre=CANONICAL_CENTRES[usual_position])
    moved_spec = move_spec(usual_spec, CANONICAL_CENTRES[moved_position])
    usual = PositionSample(
        render_shape(usual_spec),
        label,
        usual_position,
        usual_spec.centre,
        group_name(label, usual_position),
        "paired counterfactual",
        seed,
        usual_spec,
    )
    moved = PositionSample(
        render_shape(moved_spec),
        label,
        moved_position,
        moved_spec.centre,
        group_name(label, moved_position),
        "paired counterfactual",
        seed,
        moved_spec,
    )
    return usual, moved


def make_counterfactual_pairs(count: int) -> list[tuple[PositionSample, PositionSample]]:
    pairs: list[tuple[PositionSample, PositionSample]] = []
    for index in range(count):
        pairs.append(fixed_pair(0, "lower_left", "upper_right", SEED + 80_000 + index))
        pairs.append(fixed_pair(1, "upper_right", "lower_left", SEED + 81_000 + index))
    return pairs


def counterfactual_flip_rate(model: torch.nn.Module, pairs: list[tuple[PositionSample, PositionSample]]) -> float:
    flips = 0
    for usual, moved in pairs:
        predictions = evaluate_model(model, [usual, moved])["predictions"]
        flips += int(predictions[0] != predictions[1])
    return flips / len(pairs)


counterfactual_pairs = make_counterfactual_pairs(30)
mlp_counterfactual_flip_rate = counterfactual_flip_rate(baseline_mlp, counterfactual_pairs)
cnn_counterfactual_flip_rate = counterfactual_flip_rate(mitigated_cnn, counterfactual_pairs)
mlp_shortcut_gap = mlp_common_accuracy - mlp_crossed_accuracy
cnn_shortcut_gap = cnn_common_accuracy - cnn_crossed_accuracy

performance_metrics = []
risk_metrics = []


In [ ]:
moon_lower_left, moon_upper_right = fixed_pair(0, "lower_left", "upper_right", SEED + 92_001)
star_upper_right, star_lower_left = fixed_pair(1, "upper_right", "lower_left", SEED + 92_011)
selected_counterfactual_samples = [moon_lower_left, moon_upper_right, star_upper_right, star_lower_left]
selected_mlp_eval = evaluate_model(baseline_mlp, selected_counterfactual_samples)
selected_cnn_eval = evaluate_model(mitigated_cnn, selected_counterfactual_samples)
mlp_cf_predictions = [int(value) for value in selected_mlp_eval["predictions"]]
cnn_cf_predictions = [int(value) for value in selected_cnn_eval["predictions"]]
cf_scores = {"MLP": selected_mlp_eval["star_score"], "CNN": selected_cnn_eval["star_score"]}
selected_mlp_correct_probs = [float(value) for value in selected_mlp_eval["correct_prob"]]
selected_cnn_correct_probs = [float(value) for value in selected_cnn_eval["correct_prob"]]

fig, axes = plt.subplots(2, 2, figsize=(11.4, 7.4), constrained_layout=True)
fig.suptitle("Same shape, different position, different prediction", fontsize=19, fontweight="bold")
hero_panels = [
    (axes[0, 0], "Moon lower-left", moon_lower_left, 0),
    (axes[0, 1], "Same moon upper-right", moon_upper_right, 1),
    (axes[1, 0], "Star upper-right", star_upper_right, 2),
    (axes[1, 1], "Same star lower-left", star_lower_left, 3),
]
for axis, title, sample, idx in hero_panels:
    axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(f"{title}\nTRUE: {CLASS_NAMES[sample.label]} | POSITION: {sample.position.replace('_', '-')}", fontsize=11.5)
    mlp_pred = mlp_cf_predictions[idx]
    cnn_pred = cnn_cf_predictions[idx]
    mlp_ok = mlp_pred == sample.label
    cnn_ok = cnn_pred == sample.label
    mlp_score = cf_scores["MLP"][idx] if mlp_pred == 1 else 1.0 - cf_scores["MLP"][idx]
    cnn_score = cf_scores["CNN"][idx] if cnn_pred == 1 else 1.0 - cf_scores["CNN"][idx]
    badge_specs = [
        (0.04, f"MLP: {CLASS_NAMES[mlp_pred]}, score {mlp_score:.2f} {'OK' if mlp_ok else 'FAIL'}", MODEL_COLOURS["MLP"], mlp_ok),
        (0.55, f"CNN: {CLASS_NAMES[cnn_pred]}, score {cnn_score:.2f} {'OK' if cnn_ok else 'FAIL'}", MODEL_COLOURS["CNN"], cnn_ok),
    ]
    for x0, text, colour, ok in badge_specs:
        axis.text(
            x0,
            -0.13,
            text,
            transform=axis.transAxes,
            fontsize=10.5,
            color="white",
            fontweight="bold",
            bbox={"boxstyle": "round,pad=0.35", "facecolor": colour if ok else "#B23A48", "edgecolor": "none", "alpha": 0.95},
        )
fig.text(0.5, -0.02, "The MLP learned where, not what.", ha="center", fontsize=14, fontweight="bold", color=MODEL_COLOURS["MLP"])
show_fig(fig)

selected_mlp_moon_counterfactual_flipped = mlp_cf_predictions[0] != mlp_cf_predictions[1]
selected_cnn_moon_counterfactual_stable = cnn_cf_predictions[0] == cnn_cf_predictions[1] == 0
selected_mlp_star_counterfactual_flipped = mlp_cf_predictions[2] != mlp_cf_predictions[3]
selected_cnn_star_counterfactual_stable = cnn_cf_predictions[2] == cnn_cf_predictions[3] == 1

selected_mlp_moon_usual_correct_prob = selected_mlp_correct_probs[0]
selected_mlp_moon_moved_wrong_prob = float(cf_scores["MLP"][1])
selected_mlp_star_usual_correct_prob = selected_mlp_correct_probs[2]
selected_mlp_star_moved_wrong_prob = float(1.0 - cf_scores["MLP"][3])
selected_cnn_moon_moved_correct_prob = selected_cnn_correct_probs[1]
selected_cnn_star_moved_correct_prob = selected_cnn_correct_probs[3]


## 4. Act I behavioural probe: movement animation and confidence path

The strongest evidence is behavioural: move the same object and watch the score.

**Nuisance orbit.** For a fixed object \((y,a)\), movement creates

\[
\mathcal{O}_N(y,a) = \{R(y,a,n): n \in N\}.
\]

A shape-faithful model should stay stable as nuisance factors change.

In [ ]:
def sample_from_spec(spec: ShapeSpec, centre: tuple[int, int], split: str = "scan") -> PositionSample:
    moved = move_spec(spec, centre)
    position = position_name(centre)
    return PositionSample(render_shape(moved), moved.label, position, centre, group_name(moved.label, position), split, moved.seed, moved)


fixed_moon_spec = moon_lower_left.spec
fixed_star_spec = star_upper_right.spec


def interpolate_centres(start: tuple[int, int], end: tuple[int, int], steps: int = 21) -> list[tuple[int, int]]:
    centres = []
    for alpha in np.linspace(0, 1, steps):
        x = int(round((1.0 - alpha) * start[0] + alpha * end[0]))
        y = int(round((1.0 - alpha) * start[1] + alpha * end[1]))
        centres.append((x, y))
    return centres


def movement_samples(spec: ShapeSpec, start: tuple[int, int], end: tuple[int, int]) -> list[PositionSample]:
    return [sample_from_spec(spec, centre, split="movement path") for centre in interpolate_centres(start, end)]


def movement_scores(model: torch.nn.Module, spec: ShapeSpec, start: tuple[int, int], end: tuple[int, int]) -> np.ndarray:
    return evaluate_model(model, movement_samples(spec, start, end))["star_score"]


movement_progress = np.linspace(0, 1, 21)
moon_movement_samples = movement_samples(fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"])
star_movement_samples = movement_samples(fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"])
movement_path_results = {
    "moon_mlp": movement_scores(baseline_mlp, fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"]),
    "moon_cnn": movement_scores(mitigated_cnn, fixed_moon_spec, CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"]),
    "star_mlp": movement_scores(baseline_mlp, fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"]),
    "star_cnn": movement_scores(mitigated_cnn, fixed_star_spec, CANONICAL_CENTRES["upper_right"], CANONICAL_CENTRES["lower_left"]),
}


def first_crossing(progress: np.ndarray, scores: np.ndarray, starts_above: bool) -> float | None:
    if starts_above:
        crossed = np.where(scores < 0.5)[0]
    else:
        crossed = np.where(scores >= 0.5)[0]
    return None if len(crossed) == 0 else float(progress[int(crossed[0])])


fig = plt.figure(figsize=(13.4, 7.6), constrained_layout=True)
grid = fig.add_gridspec(3, 10, height_ratios=[2.3, 0.95, 0.95])
fig.suptitle("How far must the object move before the model changes its mind?", fontsize=18, fontweight="bold")
for axis, title, mlp_key, cnn_key, starts_above in [
    (fig.add_subplot(grid[0, :5]), "Same moon: lower-left → upper-right", "moon_mlp", "moon_cnn", False),
    (fig.add_subplot(grid[0, 5:]), "Same star: upper-right → lower-left", "star_mlp", "star_cnn", True),
]:
    axis.plot(movement_progress, movement_path_results[mlp_key], label="MLP", color=MODEL_COLOURS["MLP"], linewidth=2.6)
    axis.plot(movement_progress, movement_path_results[cnn_key], label="CNN", color=MODEL_COLOURS["CNN"], linewidth=2.6)
    axis.axhline(0.5, color="black", linestyle="--", linewidth=1.2, label="decision threshold")
    crossing = first_crossing(movement_progress, movement_path_results[mlp_key], starts_above=starts_above)
    if crossing is not None:
        axis.axvline(crossing, color=MODEL_COLOURS["MLP"], linestyle=":", linewidth=2)
        axis.text(crossing + 0.02, 0.08, f"MLP crosses at {crossing * 100:.0f}%", color=MODEL_COLOURS["MLP"], fontsize=10)
    axis.set_title(title)
    axis.set_xlabel("movement progress")
    axis.set_ylabel("predicted probability of star")
    axis.set_ylim(-0.03, 1.03)
    axis.grid(alpha=0.18)
    axis.legend(frameon=False, loc="center right")

thumbnail_indices = [0, 5, 10, 15, 20]
for row, samples, mlp_key, cnn_key, label in [
    (1, moon_movement_samples, "moon_mlp", "moon_cnn", "moon path"),
    (2, star_movement_samples, "star_mlp", "star_cnn", "star path"),
]:
    for col, sample_index in enumerate(thumbnail_indices):
        axis = fig.add_subplot(grid[row, col * 2 : col * 2 + 2])
        sample = samples[sample_index]
        axis.imshow(sample.image, cmap="gray", vmin=0, vmax=255)
        axis.set_xticks([])
        axis.set_yticks([])
        axis.set_title(
            f"{label} {movement_progress[sample_index] * 100:.0f}%\n"
            f"MLP {movement_path_results[mlp_key][sample_index]:.2f} | CNN {movement_path_results[cnn_key][sample_index]:.2f}",
            fontsize=8.5,
        )
fig.text(
    0.5,
    -0.02,
    "MLP crosses the decision boundary as the object moves. CNN remains stable on this controlled path.",
    ha="center",
    fontsize=12,
)
show_fig(fig)


In [ ]:
from matplotlib.backends.backend_agg import FigureCanvasAgg


def pil_font(size: int, *, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Helvetica Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Helvetica.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size)
        except OSError:
            continue
    return ImageFont.load_default()


font_title = pil_font(44, bold=True)
font_subtitle = pil_font(27, bold=True)
font_body = pil_font(22)
font_body_bold = pil_font(24, bold=True)
font_small_bold = pil_font(18, bold=True)
animation_embeds: dict[str, bytes] = {}


def gif_bytes(frames: list[Image.Image], duration_ms: int = 130) -> bytes:
    """Encode animation frames as an in-memory GIF."""
    buffer = BytesIO()
    rgb_frames = [frame.convert("RGB") for frame in frames]
    rgb_frames[0].save(buffer, format="GIF", save_all=True, append_images=rgb_frames[1:], duration=duration_ms, loop=0)
    return buffer.getvalue()


def show_gif(frames: list[Image.Image], name: str, duration_ms: int = 130) -> None:
    """Display an animation inline as a self-contained base64 GIF."""
    data = gif_bytes(frames, duration_ms=duration_ms)
    animation_embeds[name] = data
    encoded = base64.b64encode(data).decode("ascii")
    display(HTML(
        f"<figure style='margin: 0 0 16px 0;'>"
        f"<img src='data:image/gif;base64,{encoded}' style='max-width:100%; border:1px solid #ddd'/>"
        f"<figcaption style='font-size:13px; font-weight:600'>{name}</figcaption>"
        f"</figure>"
    ))
    return None


def show_gif_or_warn(frames: list[Image.Image], gif_name: str, _mp4_name: str | None = None, *, fps: int = 8, duration_ms: int = 130) -> None:
    del fps, _mp4_name
    show_gif(frames, gif_name, duration_ms=duration_ms)
    return None


def draw_pil_confidence_bar(
    draw: ImageDraw.ImageDraw,
    xy: tuple[int, int],
    width: int,
    score: float,
    colour: tuple[int, int, int],
    label: str,
    *,
    font: ImageFont.ImageFont,
) -> None:
    x, y = xy
    draw.rounded_rectangle((x, y, x + width, y + 34), radius=9, fill=(238, 238, 238), outline=(195, 195, 195), width=2)
    draw.rounded_rectangle((x, y, x + int(width * score), y + 34), radius=9, fill=colour)
    draw.text((x, y - 30), label, fill=(30, 30, 30), font=font)


def make_confidence_bar(draw: ImageDraw.ImageDraw, xy: tuple[int, int], width: int, score: float, model_name: str, colour: tuple[int, int, int]) -> None:
    pred = int(score >= 0.5)
    label = f"{model_name} P(star) {score:.2f} -> {CLASS_NAMES[pred]}"
    draw_pil_confidence_bar(draw, xy, width, float(score), colour, label, font=font_small_bold)


def star_probability_label(score: float) -> str:
    return f"star {score:.2f}" if score >= 0.5 else f"moon {1.0 - score:.2f}"


def draw_score_bar(draw: ImageDraw.ImageDraw, xy: tuple[int, int], width: int, score: float, colour: tuple[int, int, int], label: str) -> None:
    x, y = xy
    draw.rounded_rectangle((x, y, x + width, y + 28), radius=8, fill=(238, 238, 238), outline=(190, 190, 190))
    draw.rounded_rectangle((x, y, x + int(width * score), y + 28), radius=8, fill=colour)
    draw.text((x, y - 24), f"{label} P(star) {score:.2f} -> {star_probability_label(score)}", fill=(30, 30, 30), font=font_small_bold)


def figure_to_image(fig: plt.Figure) -> Image.Image:
    canvas = FigureCanvasAgg(fig)
    canvas.draw()
    array = np.asarray(canvas.buffer_rgba())
    image = Image.fromarray(array).convert("RGB")
    plt.close(fig)
    return image


def confidence_frame(sample: PositionSample, mlp_score: float, cnn_score: float, progress: float, title: str, mlp_crossing: float | None) -> Image.Image:
    frame = Image.new("RGB", (900, 520), (250, 250, 250))
    draw = ImageDraw.Draw(frame)
    draw.text((28, 22), title, fill=(25, 25, 25), font=font_subtitle)
    draw.text((28, 58), f"movement progress: {progress:.2f}", fill=(80, 80, 80), font=font_body)
    frame.paste(sample.image.resize((315, 315), Image.Resampling.LANCZOS).convert("RGB"), (42, 118))
    draw_score_bar(draw, (410, 170), 380, mlp_score, (196, 78, 82), "MLP")
    draw_score_bar(draw, (410, 265), 380, cnn_score, (46, 139, 87), "CNN")
    draw.line((410, 365, 790, 365), fill=(35, 35, 35), width=3)
    draw.ellipse((410 + int(380 * progress) - 8, 357, 410 + int(380 * progress) + 8, 373), fill=(20, 20, 20))
    draw.text((410, 385), "lower-left", fill=(75, 75, 75), font=font_small_bold)
    draw.text((685, 385), "upper-right", fill=(75, 75, 75), font=font_small_bold)
    if mlp_crossing is not None and progress >= mlp_crossing:
        draw.rounded_rectangle((410, 425, 790, 470), radius=10, fill=(255, 232, 232), outline=(196, 78, 82), width=2)
        draw.text((428, 438), f"MLP crossed its boundary at {mlp_crossing:.2f}", fill=(140, 40, 45), font=font_small_bold)
    return frame


moon_crossing = first_crossing(movement_progress, movement_path_results["moon_mlp"], starts_above=False)
star_crossing = first_crossing(movement_progress, movement_path_results["star_mlp"], starts_above=True)
moon_frames = [
    confidence_frame(sample, float(movement_path_results["moon_mlp"][index]), float(movement_path_results["moon_cnn"][index]), float(movement_progress[index]), "Same moon. Same pixels. Different place. Different answer.", moon_crossing)
    for index, sample in enumerate(moon_movement_samples)
]
star_frames = [
    confidence_frame(sample, float(movement_path_results["star_mlp"][index]), float(movement_path_results["star_cnn"][index]), float(movement_progress[index]), "Same star moves: upper-right -> lower-left", star_crossing)
    for index, sample in enumerate(star_movement_samples)
]
show_gif(moon_frames, "anim_moon_moves_confidence.gif")
show_gif(star_frames, "anim_star_moves_confidence.gif")

## 5. Act I data audit: the answer was already encoded in position

> **The answer was already encoded in object address.**

Before diagnosing the neural networks, we inspect the biased exam itself.

In [ ]:
def position_feature_matrix(samples: list[PositionSample]) -> np.ndarray:
    centres = np.array([sample.centre for sample in samples], dtype=np.float64)
    lower_left = np.array(CANONICAL_CENTRES["lower_left"], dtype=np.float64)
    upper_right = np.array(CANONICAL_CENTRES["upper_right"], dtype=np.float64)
    x = centres[:, 0]
    y = centres[:, 1]
    return np.column_stack([
        x,
        y,
        x - y,
        np.linalg.norm(centres - lower_left, axis=1),
        np.linalg.norm(centres - upper_right, axis=1),
    ])


def labels_for_samples(samples: list[PositionSample]) -> np.ndarray:
    return np.array([sample.label for sample in samples], dtype=np.int64)


def position_only_predictions(samples: list[PositionSample]) -> np.ndarray:
    features = position_feature_matrix(samples)
    return (features[:, 2] > 0).astype(np.int64)


def accuracy_from_predictions(predictions: np.ndarray, labels: np.ndarray) -> float:
    return float(np.mean(np.asarray(predictions) == np.asarray(labels)))


data_first_position_results = {
    "biased train": accuracy_from_predictions(position_only_predictions(biased_train_samples), labels_for_samples(biased_train_samples)),
    "IID validation": accuracy_from_predictions(position_only_predictions(iid_validation_samples), labels_for_samples(iid_validation_samples)),
    "balanced OOD audit": accuracy_from_predictions(position_only_predictions(ood_audit_samples), labels_for_samples(ood_audit_samples)),
    "uniform-position audit": accuracy_from_predictions(position_only_predictions(uniform_audit_samples), labels_for_samples(uniform_audit_samples)),
}

train_features = position_feature_matrix(biased_train_samples)
train_labels = labels_for_samples(biased_train_samples)
iid_features = position_feature_matrix(iid_validation_samples)

position_audit_examples = [
    biased_train_samples[0],
    next(sample for sample in biased_train_samples if sample.label == 1),
    iid_validation_samples[0],
    next(sample for sample in iid_validation_samples if sample.label == 1),
]
example_tiles = []
for sample in position_audit_examples:
    tile = sample.image.resize((116, 116), Image.Resampling.LANCZOS).convert("RGB")
    example_tiles.append(tile)
example_strip = Image.new("RGB", (232, 232), (250, 250, 250))
for index, tile in enumerate(example_tiles):
    example_strip.paste(tile, ((index % 2) * 116, (index // 2) * 116))

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.9), constrained_layout=True)
fig.suptitle("The answer was already encoded in object address", fontsize=18.5, fontweight="bold")
fig.text(0.5, 0.91, "The biased exam can be solved without looking at shape.", ha="center", fontsize=13, color="#555555")

axes[0].imshow(example_strip)
axes[0].set_title("A. Looks like shape recognition", fontsize=13, fontweight="bold")
axes[0].set_xticks([])
axes[0].set_yticks([])

scatter_axis = axes[1]
for label in [0, 1]:
    selected = train_labels == label
    scatter_axis.scatter(train_features[selected, 0], train_features[selected, 1], s=24, alpha=0.62, color=CLASS_COLOURS[label], label=CLASS_NAMES[label])
scatter_axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[2]), LOWER_LEFT_REGION[1] - LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[3] - LOWER_LEFT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[0], linewidth=2.1))
scatter_axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[2]), UPPER_RIGHT_REGION[1] - UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[3] - UPPER_RIGHT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[1], linewidth=2.1))
scatter_axis.invert_yaxis()
scatter_axis.set_xlabel("object centre x", fontsize=11)
scatter_axis.set_ylabel("object centre y", fontsize=11)
scatter_axis.set_title("B. Centres separate by label", fontsize=13, fontweight="bold")
scatter_axis.legend(frameon=False, loc="lower left", fontsize=11)
style_axes(scatter_axis)

hist_axis = axes[2]
for label in [0, 1]:
    selected = train_labels == label
    hist_axis.hist(train_features[selected, 2], bins=18, alpha=0.74, color=CLASS_COLOURS[label], label=CLASS_NAMES[label])
hist_axis.axvline(0, color="#333333", linestyle="--", linewidth=1.6)
hist_axis.text(0, hist_axis.get_ylim()[1] * 0.88, " threshold", fontsize=11, fontweight="bold")
hist_axis.set_xlabel("centre_x - centre_y", fontsize=11)
hist_axis.set_ylabel("count", fontsize=11)
hist_axis.set_title("C. One coordinate contrast is enough", fontsize=13, fontweight="bold")
hist_axis.legend(frameon=False, fontsize=11)
style_axes(hist_axis)
fig.text(0.5, -0.02, "The label was already encoded in object address.", ha="center", fontsize=12.4, fontweight="bold")
show_fig(fig)


## 6. Act I silly shortcut model: nearest neighbour without shape

In [ ]:
def fit_standardisation(features: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = features.mean(axis=0)
    std = features.std(axis=0) + 1e-8
    return mean, std


def nearest_indices_from_features(train_features: np.ndarray, query_features: np.ndarray, k: int = 3) -> np.ndarray:
    mean, std = fit_standardisation(train_features)
    train_scaled = (train_features - mean) / std
    query_scaled = (query_features - mean) / std
    distances = np.linalg.norm(query_scaled[:, None, :] - train_scaled[None, :, :], axis=2)
    return np.argsort(distances, axis=1)[:, :k]


def nearest_neighbour_predict(train_features: np.ndarray, train_labels: np.ndarray, query_features: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    nearest = nearest_indices_from_features(train_features, query_features, k=1)[:, 0]
    return train_labels[nearest], nearest


act1_position_train_features = position_feature_matrix(biased_train_samples)
act1_position_iid_features = position_feature_matrix(iid_validation_samples)
act1_position_uniform_features = position_feature_matrix(uniform_audit_samples)
act1_position_ood_features = position_feature_matrix(ood_audit_samples)
act1_position_train_labels = labels_for_samples(biased_train_samples)

act1_nn_iid_predictions, _ = nearest_neighbour_predict(
    act1_position_train_features,
    act1_position_train_labels,
    act1_position_iid_features,
)
act1_nn_uniform_predictions, _ = nearest_neighbour_predict(
    act1_position_train_features,
    act1_position_train_labels,
    act1_position_uniform_features,
)
act1_nn_ood_predictions, _ = nearest_neighbour_predict(
    act1_position_train_features,
    act1_position_train_labels,
    act1_position_ood_features,
)
act1_nn_results = {
    "IID validation": accuracy_from_predictions(act1_nn_iid_predictions, labels_for_samples(iid_validation_samples)),
    "balanced/uniform audit": accuracy_from_predictions(act1_nn_ood_predictions, labels_for_samples(ood_audit_samples)),
    "uniform-position audit": accuracy_from_predictions(act1_nn_uniform_predictions, labels_for_samples(uniform_audit_samples)),
}

act1_nn_query = moon_upper_right
act1_nn_query_features = position_feature_matrix([act1_nn_query])
act1_nn_indices = nearest_indices_from_features(act1_position_train_features, act1_nn_query_features, k=3)[0]
act1_nn_prediction = int(np.bincount(labels_for_samples([biased_train_samples[int(index)] for index in act1_nn_indices])).argmax())

fig = plt.figure(figsize=(13.6, 5.0), constrained_layout=True)
grid = fig.add_gridspec(1, 2, width_ratios=[0.88, 1.42])
fig.suptitle("A silly position rule passes the biased exam", fontsize=18.5, fontweight="bold")
fig.text(0.5, 0.91, "A non-neural shortcut model can look excellent if the test repeats the leak.", ha="center", fontsize=13, color="#555555")

card_axis = fig.add_subplot(grid[0, 0])
card_axis.axis("off")
card_axis.add_patch(plt.Rectangle((0.05, 0.10), 0.90, 0.78, facecolor="#FAFAFA", edgecolor="#333333", linewidth=1.5, transform=card_axis.transAxes))
card_axis.text(0.50, 0.80, "Position-only rule", ha="center", fontsize=15, fontweight="bold", transform=card_axis.transAxes)
card_axis.text(0.50, 0.72, "Predict star if centre_x - centre_y > 0", ha="center", fontsize=11.5, color="#555555", transform=card_axis.transAxes)
for row, (name, accuracy) in enumerate(data_first_position_results.items()):
    y = 0.58 - row * 0.12
    colour = "#2E8B57" if accuracy >= 0.95 else "#D89C21" if accuracy >= 0.45 else "#B23A48"
    card_axis.text(0.12, y, name, fontsize=11.2, transform=card_axis.transAxes)
    card_axis.text(0.88, y, f"{accuracy * 100:.1f}%", ha="right", fontsize=14.2, fontweight="bold", color=colour, transform=card_axis.transAxes)
card_axis.text(0.10, 0.12, "Shape is not required to pass the biased exam.", fontsize=11.4, fontweight="bold", transform=card_axis.transAxes)

strip_axis = fig.add_subplot(grid[0, 1])
strip_axis.axis("off")
strip_axis.set_title("Nearest neighbour without shape: using position features only", fontsize=13.5, fontweight="bold")
strip = Image.new("RGB", (600, 170), (250, 250, 250))
draw = ImageDraw.Draw(strip)
items = [("query", act1_nn_query)] + [(f"NN {col}", biased_train_samples[int(index)]) for col, index in enumerate(act1_nn_indices, start=1)]
for col, (title, sample) in enumerate(items):
    x0 = 10 + col * 145
    strip.paste(sample.image.resize((110, 110), Image.Resampling.LANCZOS).convert("RGB"), (x0, 20))
    draw.text((x0, 132), title, fill=(30, 30, 30), font=font_small_bold)
    draw.text((x0, 150), CLASS_NAMES[sample.label], fill=tuple(int(CLASS_COLOURS[sample.label].lstrip("#")[i:i+2], 16) for i in (0, 2, 4)), font=font_small_bold)
strip_axis.imshow(strip)
strip_axis.text(0.02, -0.02, f"prediction: {CLASS_NAMES[act1_nn_prediction]} | true label: {CLASS_NAMES[act1_nn_query.label]}", fontsize=12.4, fontweight="bold", color=CLASS_COLOURS[act1_nn_prediction], transform=strip_axis.transAxes)
strip_axis.text(0.02, -0.15, "This model never looked at shape.", fontsize=11.5, fontweight="bold", transform=strip_axis.transAxes)
strip_axis.text(0.58, -0.02, f"position-NN IID: {act1_nn_results['IID validation'] * 100:.1f}%", fontsize=11.5, fontweight="bold", transform=strip_axis.transAxes)
strip_axis.text(0.58, -0.15, f"audit: {act1_nn_results['balanced/uniform audit'] * 100:.1f}% | uniform: {act1_nn_results['uniform-position audit'] * 100:.1f}%", fontsize=11.5, color="#B23A48", fontweight="bold", transform=strip_axis.transAxes)
show_fig(fig)

nearest_neighbour_shortcut_results = {
    "act1_position_query_prediction": act1_nn_prediction,
    "act1_position_query_true_label": act1_nn_query.label,
    "act1_position_iid_accuracy": act1_nn_results["IID validation"],
    "act1_position_uniform_accuracy": act1_nn_results["uniform-position audit"],
}


## 7. Act I response geometry: where does the model believe stars live?

The movement path gave one counterfactual. The response map scans the same object across the canvas and asks where the model assigns star-like scores.

In [ ]:
def render_morph_image(alpha: float, centre: tuple[int, int], seed: int) -> Image.Image:
    """Render a controlled moon-to-star morph at a fixed address."""
    alpha = float(np.clip(alpha, 0.0, 1.0))
    moon_spec = make_shape_spec(0, FREE_REGION, seed, fixed_centre=centre)
    star_spec = ShapeSpec(
        label=1,
        centre=centre,
        seed=seed,
        size=moon_spec.size,
        rotation=moon_spec.rotation + 8.0,
        brightness=moon_spec.brightness,
        stroke=moon_spec.stroke,
        crescent=moon_spec.crescent,
    )
    local_rng = rng_for(seed + 99)
    yy, xx = np.mgrid[0:IMAGE_SIZE, 0:IMAGE_SIZE]
    texture = 38.0 + 1.8 * np.sin(xx / 11.0) + 1.3 * np.cos(yy / 13.0) + local_rng.normal(0.0, 1.3, (IMAGE_SIZE, IMAGE_SIZE))
    moon_mask = np.asarray(object_mask(moon_spec), dtype=np.float32) / 255.0
    star_mask = np.asarray(object_mask(star_spec), dtype=np.float32) / 255.0
    moon_weight = (1.0 - alpha) ** 3
    star_weight = alpha ** 3
    combined = np.maximum(moon_weight * moon_mask, star_weight * star_mask)
    if float(combined.max()) > 1e-8:
        combined = combined / float(combined.max())
    foreground_alpha = np.clip((combined - 0.12) / 0.58, 0.0, 1.0)
    image = texture * (1.0 - foreground_alpha) + float(moon_spec.brightness) * foreground_alpha
    return Image.fromarray(np.clip(image, 0, 255).astype(np.uint8), "L")


def score_image(model: torch.nn.Module, image: Image.Image) -> float:
    array = np.asarray(image.convert("L").resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
    x = torch.tensor(array[None, None, :, :], dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        return float(torch.softmax(model(x), dim=1)[0, 1].item())


morph_alphas = np.linspace(0, 1, 21)
shape_morph_results: dict[str, dict[str, np.ndarray]] = {}
for position_name_key, centre in CANONICAL_CENTRES.items():
    morph_images = [render_morph_image(float(alpha), centre, SEED + 93_000) for alpha in morph_alphas]
    shape_morph_results[position_name_key] = {
        "mlp": np.array([score_image(baseline_mlp, image) for image in morph_images]),
        "cnn": np.array([score_image(mitigated_cnn, image) for image in morph_images]),
    }

summary_lines = [
    "### Shape morph diagnostic prepared",
    "The morph animation keeps the object address fixed and changes only the moon-to-star shape factor.",
    "This is used as supporting behavioural evidence; movement remains the main Act I reveal.",
]
display(Markdown("\n\n".join(summary_lines)))


In [ ]:
SCAN_VALUES = np.linspace(22, 106, 17).astype(int)


def position_response(model: torch.nn.Module, spec: ShapeSpec) -> np.ndarray:
    samples = [sample_from_spec(spec, (int(x), int(y))) for y in SCAN_VALUES for x in SCAN_VALUES]
    scores = evaluate_model(model, samples)["star_score"]
    return scores.reshape(len(SCAN_VALUES), len(SCAN_VALUES))


moon_position_response_mlp = position_response(baseline_mlp, fixed_moon_spec)
star_position_response_mlp = position_response(baseline_mlp, fixed_star_spec)
moon_position_response_cnn = position_response(mitigated_cnn, fixed_moon_spec)
star_position_response_cnn = position_response(mitigated_cnn, fixed_star_spec)


def sensitivity_index(matrix: np.ndarray) -> float:
    return float(np.max(matrix) - np.min(matrix))


def shape_consistency(matrix: np.ndarray, true_label: int) -> float:
    predictions = (matrix >= 0.5).astype(int)
    return float(np.mean(predictions == true_label))


position_response_metrics = {
    "mlp_moon_psi": sensitivity_index(moon_position_response_mlp),
    "mlp_star_psi": sensitivity_index(star_position_response_mlp),
    "cnn_moon_psi": sensitivity_index(moon_position_response_cnn),
    "cnn_star_psi": sensitivity_index(star_position_response_cnn),
    "mlp_moon_shape_consistency": shape_consistency(moon_position_response_mlp, 0),
    "mlp_star_shape_consistency": shape_consistency(star_position_response_mlp, 1),
    "cnn_moon_shape_consistency": shape_consistency(moon_position_response_cnn, 0),
    "cnn_star_shape_consistency": shape_consistency(star_position_response_cnn, 1),
}
mlp_position_sensitivity = float(np.mean([position_response_metrics["mlp_moon_psi"], position_response_metrics["mlp_star_psi"]]))
cnn_position_sensitivity = float(np.mean([position_response_metrics["cnn_moon_psi"], position_response_metrics["cnn_star_psi"]]))
mlp_shape_consistency = float(np.mean([position_response_metrics["mlp_moon_shape_consistency"], position_response_metrics["mlp_star_shape_consistency"]]))
cnn_shape_consistency = float(np.mean([position_response_metrics["cnn_moon_shape_consistency"], position_response_metrics["cnn_star_shape_consistency"]]))

response_panels = [
    ("MLP fixed moon", moon_position_response_mlp, 0, "star score rises\nin upper-right"),
    ("MLP fixed star", star_position_response_mlp, 1, "star score falls\nin lower-left"),
    ("CNN fixed moon", moon_position_response_cnn, 0, "stable moon\nprediction"),
    ("CNN fixed star", star_position_response_cnn, 1, "stable star\nprediction"),
]
fig, axes = plt.subplots(2, 2, figsize=(11.4, 9.2), constrained_layout=True)
fig.suptitle("Where does the model believe stars live?", fontsize=18, fontweight="bold")
for axis, (title, matrix, true_label, annotation) in zip(axes.flat, response_panels, strict=True):
    im = axis.imshow(
        matrix,
        cmap="RdYlBu_r",
        vmin=0,
        vmax=1,
        extent=[SCAN_VALUES[0], SCAN_VALUES[-1], SCAN_VALUES[-1], SCAN_VALUES[0]],
        interpolation="bicubic",
    )
    axis.contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.5], colors="black", linewidths=1.4)
    axis.add_patch(plt.Rectangle((LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[2]), LOWER_LEFT_REGION[1]-LOWER_LEFT_REGION[0], LOWER_LEFT_REGION[3]-LOWER_LEFT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[0], linewidth=2))
    axis.add_patch(plt.Rectangle((UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[2]), UPPER_RIGHT_REGION[1]-UPPER_RIGHT_REGION[0], UPPER_RIGHT_REGION[3]-UPPER_RIGHT_REGION[2], fill=False, edgecolor=CLASS_COLOURS[1], linewidth=2))
    axis.text(LOWER_LEFT_REGION[0] + 1, LOWER_LEFT_REGION[3] - 2, "moon territory", color=CLASS_COLOURS[0], fontsize=9.2, fontweight="bold")
    axis.text(UPPER_RIGHT_REGION[0] + 1, UPPER_RIGHT_REGION[3] - 2, "star territory", color=CLASS_COLOURS[1], fontsize=9.2, fontweight="bold")
    psi = sensitivity_index(matrix)
    consistency = shape_consistency(matrix, true_label)
    axis.set_title(f"{title}\nPSI {psi:.2f}; shape consistency {consistency * 100:.0f}%")
    axis.text(0.04, 0.08, annotation, transform=axis.transAxes, fontsize=10.5, fontweight="bold", color="white", bbox={"boxstyle":"round,pad=0.35", "facecolor":"black", "alpha":0.70, "edgecolor":"none"})
    axis.set_xlabel("object centre x")
    axis.set_ylabel("object centre y")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, label="predicted probability of star")
fig.text(0.18, 0.015, "blue = moon-like score", color="#2D6F9F", fontsize=11, fontweight="bold")
fig.text(0.38, 0.015, "red = star-like score", color="#B23A48", fontsize=11, fontweight="bold")
fig.text(0.58, 0.015, "The MLP learned label geography. The CNN fixed this shortcut, not all shortcuts.", fontsize=10.5, fontweight="bold")
show_fig(fig)


In [ ]:
def response_map_path_frame(matrix: np.ndarray, sample: PositionSample, centre: tuple[int, int], progress: float, title: str) -> Image.Image:
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.0), constrained_layout=True)
    fig.suptitle(title, fontsize=15, fontweight="bold")
    axes[0].imshow(matrix, cmap="RdYlBu_r", vmin=0, vmax=1, extent=[SCAN_VALUES[0], SCAN_VALUES[-1], SCAN_VALUES[-1], SCAN_VALUES[0]], interpolation="bicubic")
    axes[0].contour(SCAN_VALUES, SCAN_VALUES, matrix, levels=[0.5], colors="black", linewidths=1.8)
    axes[0].scatter([centre[0]], [centre[1]], s=95, color="white", edgecolor="black", linewidth=2.0, zorder=6)
    axes[0].set_title("position-response map")
    axes[0].set_xlim(SCAN_VALUES[0], SCAN_VALUES[-1])
    axes[0].set_ylim(SCAN_VALUES[-1], SCAN_VALUES[0])
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[1].imshow(sample.image, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"same moon, progress {progress:.2f}")
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    return figure_to_image(fig)


mlp_response_frames = [
    response_map_path_frame(moon_position_response_mlp, sample, sample.centre, float(movement_progress[index]), "The MLP learned territory.")
    for index, sample in enumerate(moon_movement_samples)
]
cnn_response_frames = [
    response_map_path_frame(moon_position_response_cnn, sample, sample.centre, float(movement_progress[index]), "CNN: same moon remains shape-stable.")
    for index, sample in enumerate(moon_movement_samples)
]

show_gif(mlp_response_frames, "anim_response_map_path_mlp.gif", duration_ms=120)
show_gif(cnn_response_frames, "anim_response_map_path_cnn.gif", duration_ms=120)


def act1_gradient_saliency(model: torch.nn.Module, image: Image.Image, target_class: int | None = None) -> np.ndarray:
    """Fast gradient saliency for animation frames.

    This is a visual diagnostic only. The behavioural evidence remains the
    controlled counterfactual score path.
    """
    model.eval()
    array = np.asarray(image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
    x = torch.tensor(array[None, None, :, :], dtype=torch.float32, requires_grad=True)
    logits = model(x)
    if target_class is None:
        target_class = int(logits.argmax(dim=1).item())
    score = logits[0, target_class]
    model.zero_grad(set_to_none=True)
    score.backward()
    saliency = (x.grad.detach().abs()[0, 0].cpu().numpy() * (array + 0.15))
    saliency = saliency - float(saliency.min())
    return saliency / max(float(saliency.max()), 1e-8)


def heatmap_animation_frame(
    sample: PositionSample,
    index: int,
    mlp_key: str,
    cnn_key: str,
    title: str,
) -> Image.Image:
    mlp_score = float(movement_path_results[mlp_key][index])
    cnn_score = float(movement_path_results[cnn_key][index])
    mlp_saliency = act1_gradient_saliency(baseline_mlp, sample.image)
    cnn_saliency = act1_gradient_saliency(mitigated_cnn, sample.image)

    fig = plt.figure(figsize=(12.2, 6.0), constrained_layout=True)
    grid = fig.add_gridspec(2, 4, width_ratios=[1.0, 1.0, 1.0, 1.35], height_ratios=[1.0, 0.62])
    fig.suptitle(title, fontsize=17, fontweight="bold")

    for axis, panel_title, saliency in [
        (fig.add_subplot(grid[0, 0]), "same object", None),
        (fig.add_subplot(grid[0, 1]), "MLP gradient heatmap", mlp_saliency),
        (fig.add_subplot(grid[0, 2]), "CNN gradient heatmap", cnn_saliency),
    ]:
        axis.imshow(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), cmap="gray", vmin=0, vmax=255)
        if saliency is not None:
            axis.imshow(saliency, cmap="magma", alpha=0.70)
        axis.set_title(panel_title)
        axis.set_xticks([])
        axis.set_yticks([])

    score_axis = fig.add_subplot(grid[0, 3])
    bars = score_axis.barh(["MLP", "CNN"], [mlp_score, cnn_score], color=[MODEL_COLOURS["MLP"], MODEL_COLOURS["CNN"]], alpha=0.92)
    score_axis.axvline(0.5, color="black", linestyle="--", linewidth=1.0)
    score_axis.set_xlim(0, 1)
    score_axis.set_xlabel("P(star)")
    score_axis.set_title(f"progress {movement_progress[index]:.2f}")
    for bar, score in zip(bars, [mlp_score, cnn_score], strict=True):
        score_axis.text(min(score + 0.025, 0.94), bar.get_y() + bar.get_height() / 2, f"{score:.2f}", va="center", fontsize=10.5, fontweight="bold")

    path_axis = fig.add_subplot(grid[1, :])
    path_axis.plot(movement_progress, movement_path_results[mlp_key], color=MODEL_COLOURS["MLP"], linewidth=2.4, label="MLP P(star)")
    path_axis.plot(movement_progress, movement_path_results[cnn_key], color=MODEL_COLOURS["CNN"], linewidth=2.4, label="CNN P(star)")
    path_axis.scatter([movement_progress[index]], [mlp_score], color=MODEL_COLOURS["MLP"], s=70, zorder=5)
    path_axis.scatter([movement_progress[index]], [cnn_score], color=MODEL_COLOURS["CNN"], s=70, zorder=5)
    path_axis.axhline(0.5, color="black", linestyle="--", linewidth=1.0)
    path_axis.set_ylim(-0.03, 1.03)
    path_axis.set_xlabel("movement progress")
    path_axis.set_ylabel("P(star)")
    path_axis.grid(alpha=0.16)
    path_axis.legend(frameon=False, loc="best")
    fig.text(0.5, -0.02, "The heatmap helps show changing visual evidence, but the controlled score path is the diagnostic.", ha="center", fontsize=11.5, fontweight="bold")
    return figure_to_image(fig)


heatmap_frame_indices = np.linspace(0, len(moon_movement_samples) - 1, 6, dtype=int)
moon_heatmap_frames = [
    heatmap_animation_frame(
        moon_movement_samples[int(index)],
        int(index),
        "moon_mlp",
        "moon_cnn",
        "Same moon moves: heatmaps and confidence change together",
    )
    for index in heatmap_frame_indices
]
star_heatmap_frames = [
    heatmap_animation_frame(
        star_movement_samples[int(index)],
        int(index),
        "star_mlp",
        "star_cnn",
        "Same star moves: heatmaps and confidence change together",
    )
    for index in heatmap_frame_indices
]
show_gif(moon_heatmap_frames, "anim_moon_moves_heatmaps.gif", duration_ms=150)
show_gif(star_heatmap_frames, "anim_star_moves_heatmaps.gif", duration_ms=150)


def morph_frame(alpha: float, centre: tuple[int, int], title: str) -> Image.Image:
    image = render_morph_image(alpha, centre, SEED + 93_000)
    mlp_score = score_image(baseline_mlp, image)
    cnn_score = score_image(mitigated_cnn, image)
    address = "lower-left" if centre == CANONICAL_CENTRES["lower_left"] else "upper-right"
    frame = Image.new("RGB", (900, 520), (250, 250, 250))
    draw = ImageDraw.Draw(frame)
    draw.text((28, 22), title, fill=(25, 25, 25), font=font_subtitle)
    draw.text((28, 58), f"fixed address: {address} | morph alpha: {alpha:.2f}", fill=(80, 80, 80), font=font_body)
    frame.paste(image.resize((315, 315), Image.Resampling.LANCZOS).convert("RGB"), (42, 118))
    draw_score_bar(draw, (410, 170), 380, mlp_score, (196, 78, 82), "MLP")
    draw_score_bar(draw, (410, 265), 380, cnn_score, (46, 139, 87), "CNN")
    draw.line((410, 365, 790, 365), fill=(35, 35, 35), width=3)
    draw.ellipse((410 + int(380 * alpha) - 8, 357, 410 + int(380 * alpha) + 8, 373), fill=(20, 20, 20))
    draw.text((410, 385), "moon shape", fill=(75, 75, 75), font=font_small_bold)
    draw.text((685, 385), "star shape", fill=(75, 75, 75), font=font_small_bold)
    return frame


morph_animation_alphas = np.linspace(0, 1, 21)
lower_left_morph_frames = [morph_frame(float(alpha), CANONICAL_CENTRES["lower_left"], "Morph at lower-left: moon -> star") for alpha in morph_animation_alphas]
upper_right_morph_frames = [morph_frame(float(alpha), CANONICAL_CENTRES["upper_right"], "Morph at upper-right: moon -> star") for alpha in morph_animation_alphas]

show_gif(lower_left_morph_frames, "anim_morph_lower_left.gif", duration_ms=105)
show_gif(upper_right_morph_frames, "anim_morph_upper_right.gif", duration_ms=105)


def morph_heatmap_frame(alpha: float, centre: tuple[int, int], title: str) -> Image.Image:
    image = render_morph_image(alpha, centre, SEED + 93_000)
    mlp_score = score_image(baseline_mlp, image)
    cnn_score = score_image(mitigated_cnn, image)
    mlp_saliency = act1_gradient_saliency(baseline_mlp, image)
    cnn_saliency = act1_gradient_saliency(mitigated_cnn, image)

    fig = plt.figure(figsize=(11.2, 5.4), constrained_layout=True)
    grid = fig.add_gridspec(1, 4, width_ratios=[1.0, 1.0, 1.0, 1.25])
    fig.suptitle(title, fontsize=17, fontweight="bold")
    for axis, panel_title, saliency in [
        (fig.add_subplot(grid[0, 0]), "morphed object", None),
        (fig.add_subplot(grid[0, 1]), "MLP heatmap", mlp_saliency),
        (fig.add_subplot(grid[0, 2]), "CNN heatmap", cnn_saliency),
    ]:
        axis.imshow(image.resize((MODEL_SIZE, MODEL_SIZE)), cmap="gray", vmin=0, vmax=255)
        if saliency is not None:
            axis.imshow(saliency, cmap="magma", alpha=0.70)
        axis.set_title(panel_title)
        axis.set_xticks([])
        axis.set_yticks([])

    score_axis = fig.add_subplot(grid[0, 3])
    bars = score_axis.barh(["MLP", "CNN"], [mlp_score, cnn_score], color=[MODEL_COLOURS["MLP"], MODEL_COLOURS["CNN"]], alpha=0.92)
    score_axis.axvline(0.5, color="black", linestyle="--", linewidth=1.0)
    score_axis.set_xlim(0, 1)
    score_axis.set_xlabel("P(star)")
    score_axis.set_title(f"shape alpha {alpha:.2f}")
    for bar, score in zip(bars, [mlp_score, cnn_score], strict=True):
        score_axis.text(min(score + 0.025, 0.94), bar.get_y() + bar.get_height() / 2, f"{score:.2f}", va="center", fontsize=10.5, fontweight="bold")
    fig.text(0.5, -0.02, "Morph heatmaps show visual context; the score curve still answers what changed the decision.", ha="center", fontsize=11.5, fontweight="bold")
    return figure_to_image(fig)


morph_heatmap_alphas = np.linspace(0, 1, 6)
lower_left_morph_heatmap_frames = [
    morph_heatmap_frame(float(alpha), CANONICAL_CENTRES["lower_left"], "Moon-to-star morph at lower-left: heatmaps and scores")
    for alpha in morph_heatmap_alphas
]
upper_right_morph_heatmap_frames = [
    morph_heatmap_frame(float(alpha), CANONICAL_CENTRES["upper_right"], "Moon-to-star morph at upper-right: heatmaps and scores")
    for alpha in morph_heatmap_alphas
]
show_gif(lower_left_morph_heatmap_frames, "anim_morph_lower_left_heatmaps.gif", duration_ms=150)
show_gif(upper_right_morph_heatmap_frames, "anim_morph_upper_right_heatmaps.gif", duration_ms=150)


## 8. Act I intervention and re-test

The intervention is not a nicer heatmap. It changes the learning incentive and then re-tests the same behavioural probes. In this controlled setting, the CNN with shared filters, global pooling, and position augmentation is much less tied to canvas territory.

The CNN fixed this position shortcut in this controlled setup. That does not make CNNs universally safe.

## 9. Act II setup: CNN appears to work again

After Act I, it is tempting to conclude that the CNN solved the problem. That is the wrong conclusion. Now the data offers a different cheap rule: a nearly invisible background-colour statistic.

In [ ]:
# M1. Act II constants and deterministic renderer.
BASE_GREY = 0.50
ACT2_NOISE_STD = 0.006
ACT2_CUE_CHANNEL = 0
ACT2_COLOUR_SCALE = 1500.0
ACT2_TINT_DELTA_CANDIDATES = [value / 255.0 for value in [1, 2, 3, 4, 5, 6, 8, 10]]
ACT2_SEED = SEED + 220_000
ACT2_COUNTERFACTUAL_SEED = ACT2_SEED + 92_000
ACT2_MARGIN_TARGET = 6.0
ACT2_SUCCESS_MESSAGE = "Act II failed: invisible background cue did not produce decisive confidence."


@dataclass(frozen=True)
class Act2Sample:
    image: Image.Image
    label: int
    background_style: str
    seed: int
    spec: ShapeSpec


def make_base_noise(seed: int, h: int = IMAGE_SIZE, w: int = IMAGE_SIZE) -> np.ndarray:
    local_rng = np.random.default_rng(seed)
    return local_rng.normal(0.0, ACT2_NOISE_STD, size=(h, w, 3)).astype(np.float32)


def render_subtle_background(style: str, seed: int, tint_delta: float, shape: tuple[int, int] = (IMAGE_SIZE, IMAGE_SIZE)) -> np.ndarray:
    background = BASE_GREY + make_base_noise(seed, shape[0], shape[1])
    if style == "star":
        background[..., ACT2_CUE_CHANNEL] += tint_delta
    return np.clip(background, 0.0, 1.0)


def interpolate_backgrounds(bg_a: np.ndarray, bg_b: np.ndarray, alpha: float) -> np.ndarray:
    return np.clip((1.0 - alpha) * bg_a + alpha * bg_b, 0.0, 1.0)


def make_act2_spec(label: int, seed: int, centre: tuple[int, int] | None = None) -> ShapeSpec:
    local_rng = rng_for(seed)
    chosen_centre = centre if centre is not None else choose_centre(FREE_REGION, local_rng)
    return ShapeSpec(
        label=label,
        centre=chosen_centre,
        seed=seed,
        size=float(local_rng.uniform(18.0, 22.0)),
        rotation=float(local_rng.uniform(-16.0, 16.0)),
        brightness=int(local_rng.integers(225, 256)),
        stroke=1,
        crescent=float(local_rng.uniform(0.62, 0.70)),
    )


def foreground_from_shape(spec: ShapeSpec) -> tuple[np.ndarray, np.ndarray]:
    grey = np.asarray(render_shape(spec), dtype=np.float32) / 255.0
    mask = grey > 0.34
    foreground = np.clip(0.18 + 0.82 * grey, 0.0, 1.0)
    return foreground, mask


def render_act2_image_from_spec(spec: ShapeSpec, background: np.ndarray) -> Image.Image:
    foreground, mask = foreground_from_shape(spec)
    rgb = background.copy()
    rgb[mask] = np.repeat(foreground[..., None], 3, axis=2)[mask]
    return Image.fromarray(np.uint8(np.clip(rgb, 0, 1) * 255), "RGB")


def render_same_shape_with_background_style(spec: ShapeSpec, background_style: str, seed: int, tint_delta: float) -> Image.Image:
    return render_act2_image_from_spec(spec, render_subtle_background(background_style, seed, tint_delta))


def render_neutral_act2_image(spec: ShapeSpec) -> Image.Image:
    neutral_background = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), 0.35, dtype=np.float32)
    return render_act2_image_from_spec(spec, neutral_background)


def make_invisible_background_dataset(
    count_per_class: int,
    *,
    tint_delta: float,
    cue_probability: float = 1.0,
    randomise_background: bool = False,
    neutralise_background: bool = False,
    seed: int = ACT2_SEED,
    split: str = "train",
) -> list[Act2Sample]:
    samples: list[Act2Sample] = []
    local_rng = rng_for(seed)
    for label in [0, 1]:
        for index in range(count_per_class):
            sample_seed = int(seed + label * 20_000 + index)
            spec = make_act2_spec(label, sample_seed)
            if neutralise_background:
                style = "neutralised"
                image = render_neutral_act2_image(spec)
            else:
                if randomise_background:
                    style = "star" if bool(local_rng.integers(0, 2)) else "moon"
                elif local_rng.random() < cue_probability:
                    style = CLASS_NAMES[label]
                else:
                    style = CLASS_NAMES[1 - label]
                image = render_same_shape_with_background_style(spec, style, sample_seed + 991, tint_delta)
            samples.append(Act2Sample(image=image, label=label, background_style=style, seed=sample_seed, spec=spec))
    local_rng.shuffle(samples)
    return samples


ACT2_TENSOR_CACHE: dict[tuple[int, ...], tuple[torch.Tensor, torch.Tensor]] = {}


def act2_samples_to_tensors(samples: list[Act2Sample]) -> tuple[torch.Tensor, torch.Tensor]:
    cache_key = tuple(id(sample) for sample in samples)
    if cache_key in ACT2_TENSOR_CACHE:
        return ACT2_TENSOR_CACHE[cache_key]
    arrays = [np.asarray(sample.image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0 for sample in samples]
    x = torch.tensor(np.stack(arrays).transpose(0, 3, 1, 2), dtype=torch.float32)
    y = torch.tensor([sample.label for sample in samples], dtype=torch.long)
    ACT2_TENSOR_CACHE[cache_key] = (x, y)
    return x, y


def image_batch_to_tensor(image_or_batch: Image.Image | list[Image.Image]) -> tuple[torch.Tensor, bool]:
    images = image_or_batch if isinstance(image_or_batch, list) else [image_or_batch]
    arrays = [np.asarray(image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0 for image in images]
    x = torch.tensor(np.stack(arrays).transpose(0, 3, 1, 2), dtype=torch.float32)
    return x, isinstance(image_or_batch, list)

In [ ]:
# M2. Act II models, confidence metrics, and margin training.
class Act2CueCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.visual = torch.nn.Sequential(
            torch.nn.Conv2d(3, 16, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(16, 32, kernel_size=5, padding=2),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 48, kernel_size=3, padding=1),
            torch.nn.ReLU(),
            torch.nn.AdaptiveAvgPool2d(1),
        )
        self.colour_head = torch.nn.Sequential(
            torch.nn.Linear(3, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 16),
            torch.nn.ReLU(),
        )
        self.head = torch.nn.Linear(48 + 16, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        visual_features = self.visual(x).flatten(1)
        # Deliberately give the network access to global colour statistics.
        # This mirrors real acquisition colour, lighting, camera balance, batch effects, or background statistics.
        colour_stats = (x.mean(dim=(2, 3)) - 0.5) * ACT2_COLOUR_SCALE
        colour_features = self.colour_head(colour_stats)
        return self.head(torch.cat([visual_features, colour_features], dim=1))


class BackgroundMeanOnlyClassifier(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = torch.nn.Linear(3, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        colour_stats = (x.mean(dim=(2, 3)) - 0.5) * ACT2_COLOUR_SCALE
        return self.linear(colour_stats)


class Act2GrayGapCNN(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.model = GapCNN()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        grey = x.mean(dim=1, keepdim=True)
        return self.model(grey)


class LogitScaledModel(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module, scale: float) -> None:
        super().__init__()
        self.base_model = base_model
        self.scale = float(scale)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base_model(x) * self.scale


def correct_class_probability(p_star: float, label: int) -> float:
    return float(p_star if label == 1 else 1.0 - p_star)


def margin_loss(logits: torch.Tensor, y: torch.Tensor, target_margin: float = ACT2_MARGIN_TARGET) -> torch.Tensor:
    correct = logits.gather(1, y[:, None]).squeeze(1)
    wrong = logits.gather(1, (1 - y)[:, None]).squeeze(1)
    return torch.relu(target_margin - (correct - wrong)).mean()


def predict_p_star(model: torch.nn.Module, image_or_batch: Image.Image | list[Image.Image]) -> float | np.ndarray:
    x, is_batch = image_batch_to_tensor(image_or_batch)
    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(x), dim=1)[:, 1].cpu().numpy()
    return probabilities if is_batch else float(probabilities[0])


def prediction_text(p_star: float) -> str:
    return f"star ({p_star:.2f})" if p_star >= 0.5 else f"moon ({1.0 - p_star:.2f})"


def make_confidence_bar(draw: ImageDraw.ImageDraw, xy: tuple[int, int], width: int, p_star: float, label: str, colour: tuple[int, int, int]) -> None:
    x, y = xy
    draw.rounded_rectangle((x, y, x + width, y + 28), radius=8, fill=(238, 238, 238), outline=(185, 185, 185))
    draw.rounded_rectangle((x, y, x + int(width * p_star), y + 28), radius=8, fill=colour)
    draw.text((x, y - 21), f"{label} P(star) {p_star:.2f} -> {prediction_text(p_star)}", fill=(30, 30, 30))


def evaluate_act2_model(model: torch.nn.Module, samples: list[Act2Sample]) -> dict[str, Any]:
    x, y = act2_samples_to_tensors(samples)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(logits, y).item()
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)
        star_scores = probabilities[:, 1].cpu().numpy()
    labels = y.cpu().numpy()
    correct_probs = np.array([correct_class_probability(float(score), int(label)) for score, label in zip(star_scores, labels, strict=True)])
    return {
        "loss": loss,
        "accuracy": float((predictions == y).float().mean().item()),
        "mean_correct_prob": float(correct_probs.mean()),
        "min_correct_prob": float(correct_probs.min()),
        "star_score": star_scores,
        "predictions": predictions.cpu().numpy(),
        "labels": labels,
    }


def train_act2_model(
    model: torch.nn.Module,
    train_samples: list[Act2Sample],
    validation_samples: list[Act2Sample],
    *,
    epochs: int = 200,
    learning_rate: float = 2e-3,
    require_confidence: bool = False,
) -> list[dict[str, float]]:
    x_train, y_train = act2_samples_to_tensors(train_samples)
    optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    history: list[dict[str, float]] = []
    for epoch in range(1, epochs + 1):
        model.train()
        optimiser.zero_grad(set_to_none=True)
        logits = model(x_train)
        loss = torch.nn.functional.cross_entropy(logits, y_train) + 0.10 * margin_loss(logits, y_train)
        loss.backward()
        optimiser.step()
        train_eval = evaluate_act2_model(model, train_samples)
        validation_eval = evaluate_act2_model(model, validation_samples)
        history.append(
            {
                "epoch": float(epoch),
                "train_accuracy": train_eval["accuracy"],
                "validation_accuracy": validation_eval["accuracy"],
                "validation_mean_correct_prob": validation_eval["mean_correct_prob"],
                "validation_min_correct_prob": validation_eval["min_correct_prob"],
                "train_loss": train_eval["loss"],
                "validation_loss": validation_eval["loss"],
            }
        )
        if require_confidence:
            if (
                epoch >= 12
                and validation_eval["accuracy"] >= 0.995
                and validation_eval["mean_correct_prob"] >= 0.98
                and validation_eval["min_correct_prob"] >= 0.90
            ):
                break
        elif epoch >= 12 and validation_eval["accuracy"] >= 0.995:
            break
    return history


def calibrate_logit_scale(
    model: torch.nn.Module,
    validation_samples: list[Act2Sample],
    accept: Callable[[torch.nn.Module], bool],
    *,
    scales: tuple[float, ...] = (1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 16.0, 24.0, 32.0, 48.0, 64.0),
) -> tuple[torch.nn.Module, float]:
    for scale in scales:
        candidate = LogitScaledModel(model, scale)
        validation_eval = evaluate_act2_model(candidate, validation_samples)
        if validation_eval["accuracy"] >= 0.995 and accept(candidate):
            return candidate, scale
    return model, 1.0

In [ ]:
# M3. Calibrate the smallest invisible tint that gives decisive Act II confidence.
def make_act2_swap_cases(tint_delta: float) -> tuple[list[tuple[str, ShapeSpec, str, int, str]], dict[str, Image.Image]]:
    moon_spec = make_act2_spec(0, ACT2_SEED + 91_000, centre=(44, 72))
    star_spec = make_act2_spec(1, ACT2_SEED + 91_500, centre=(84, 54))
    cases = [
        ("same moon\nmoon background", moon_spec, "moon", 0, "moon_on_moon_bg"),
        ("same moon\nstar background", moon_spec, "star", 0, "moon_on_star_bg"),
        ("same star\nstar background", star_spec, "star", 1, "star_on_star_bg"),
        ("same star\nmoon background", star_spec, "moon", 1, "star_on_moon_bg"),
    ]
    seed_by_key = {
        "moon_on_moon_bg": ACT2_COUNTERFACTUAL_SEED,
        "moon_on_star_bg": ACT2_COUNTERFACTUAL_SEED,
        "star_on_star_bg": ACT2_COUNTERFACTUAL_SEED + 1,
        "star_on_moon_bg": ACT2_COUNTERFACTUAL_SEED + 1,
    }
    images = {
        key: render_same_shape_with_background_style(spec, style, seed_by_key[key], tint_delta)
        for _title, spec, style, _label, key in cases
    }
    return cases, images


def act2_swap_score_dict(model: torch.nn.Module, images: dict[str, Image.Image]) -> dict[str, float]:
    return {key: float(predict_p_star(model, image)) for key, image in images.items()}


def biased_swap_thresholds_pass(scores: dict[str, float]) -> bool:
    return (
        scores["moon_on_moon_bg"] <= 0.05
        and scores["moon_on_star_bg"] >= 0.95
        and scores["star_on_star_bg"] >= 0.95
        and scores["star_on_moon_bg"] <= 0.05
    )


def mitigated_swap_thresholds_pass(scores: dict[str, float]) -> bool:
    return (
        scores["moon_on_moon_bg"] <= 0.10
        and scores["moon_on_star_bg"] <= 0.10
        and scores["star_on_star_bg"] >= 0.90
        and scores["star_on_moon_bg"] >= 0.90
    )


def act2_ready(model: torch.nn.Module, validation_samples: list[Act2Sample], swap_images: dict[str, Image.Image]) -> bool:
    validation_eval = evaluate_act2_model(model, validation_samples)
    return (
        validation_eval["accuracy"] >= 0.995
        and validation_eval["mean_correct_prob"] >= 0.98
        and validation_eval["min_correct_prob"] >= 0.90
        and biased_swap_thresholds_pass(act2_swap_score_dict(model, swap_images))
    )


def train_background_only_for_delta(tint_delta: float) -> tuple[torch.nn.Module, list[Act2Sample], list[Act2Sample], list[dict[str, float]], dict[str, Any]]:
    train_samples = make_invisible_background_dataset(120, tint_delta=tint_delta, seed=ACT2_SEED, split="background-only train")
    validation_samples = make_invisible_background_dataset(50, tint_delta=tint_delta, seed=ACT2_SEED + 10_000, split="background-only validation")
    torch.manual_seed(ACT2_SEED + int(round(tint_delta * 255)) + 11)
    model = BackgroundMeanOnlyClassifier()
    history = train_act2_model(model, train_samples, validation_samples, epochs=160, learning_rate=2e-3, require_confidence=True)
    cases, swap_images = make_act2_swap_cases(tint_delta)
    def accept(candidate: torch.nn.Module) -> bool:
        return biased_swap_thresholds_pass(act2_swap_score_dict(candidate, swap_images))
    model, scale = calibrate_logit_scale(model, validation_samples, accept)
    evaluation = evaluate_act2_model(model, validation_samples)
    evaluation["scale"] = scale
    evaluation["swap_scores"] = act2_swap_score_dict(model, swap_images)
    return model, train_samples, validation_samples, history, evaluation


def train_cue_cnn_for_delta(tint_delta: float) -> dict[str, Any]:
    train_samples = make_invisible_background_dataset(160, tint_delta=tint_delta, seed=ACT2_SEED, split="train")
    validation_samples = make_invisible_background_dataset(60, tint_delta=tint_delta, seed=ACT2_SEED + 10_000, split="validation")
    cases, swap_images = make_act2_swap_cases(tint_delta)
    torch.manual_seed(ACT2_SEED + int(round(tint_delta * 255)) + 21)
    model = Act2CueCNN()
    history = train_act2_model(model, train_samples, validation_samples, epochs=200, learning_rate=2e-3, require_confidence=True)
    def accept(candidate: torch.nn.Module) -> bool:
        return act2_ready(candidate, validation_samples, swap_images)
    model, scale = calibrate_logit_scale(model, validation_samples, accept)
    validation_eval = evaluate_act2_model(model, validation_samples)
    swap_scores = act2_swap_score_dict(model, swap_images)
    return {
        "model": model,
        "scale": scale,
        "train_samples": train_samples,
        "validation_samples": validation_samples,
        "history": history,
        "validation_eval": validation_eval,
        "swap_cases": cases,
        "swap_images": swap_images,
        "swap_scores": swap_scores,
        "ready": act2_ready(model, validation_samples, swap_images),
    }


act2_calibration_trials: list[dict[str, Any]] = []
act2_background_only_trials: list[dict[str, Any]] = []
act2_selected_trial: dict[str, Any] | None = None
act2_background_only_model: torch.nn.Module | None = None
act2_background_only_eval: dict[str, Any] | None = None

for candidate_delta in ACT2_TINT_DELTA_CANDIDATES:
    bg_model, bg_train, bg_validation, bg_history, bg_eval = train_background_only_for_delta(candidate_delta)
    act2_background_only_trials.append(
        {
            "tint_delta": candidate_delta,
            "model": bg_model,
            "train_samples": bg_train,
            "validation_samples": bg_validation,
            "history": bg_history,
            "evaluation": bg_eval,
        }
    )
    trial = train_cue_cnn_for_delta(candidate_delta)
    trial["tint_delta"] = candidate_delta
    act2_calibration_trials.append(trial)
    if trial["ready"]:
        act2_selected_trial = trial
        act2_background_only_model = bg_model
        act2_background_only_eval = bg_eval
        break

if act2_selected_trial is None or act2_background_only_model is None or act2_background_only_eval is None:
    raise RuntimeError(ACT2_SUCCESS_MESSAGE)

ACT2_TINT_DELTA = float(act2_selected_trial["tint_delta"])
act2_biased_cnn = act2_selected_trial["model"]
act2_train_samples = act2_selected_trial["train_samples"]
act2_validation_samples = act2_selected_trial["validation_samples"]
act2_history = act2_selected_trial["history"]
act2_swap_cases = act2_selected_trial["swap_cases"]
act2_swap_images = act2_selected_trial["swap_images"]
act2_swap_scores = act2_selected_trial["swap_scores"]
act2_train_accuracy = evaluate_act2_model(act2_biased_cnn, act2_train_samples)["accuracy"]
act2_validation_eval = act2_selected_trial["validation_eval"]
act2_validation_accuracy = act2_validation_eval["accuracy"]
act2_validation_mean_correct_prob = act2_validation_eval["mean_correct_prob"]
act2_validation_min_correct_prob = act2_validation_eval["min_correct_prob"]
act2_background_only_accuracy = float(act2_background_only_eval["accuracy"])
act2_background_only_mean_correct_prob = float(act2_background_only_eval["mean_correct_prob"])
act2_background_only_min_correct_prob = float(act2_background_only_eval["min_correct_prob"])
act2_background_only_swap_scores = act2_background_only_eval["swap_scores"]
act2_tint_delta_label = f"{ACT2_TINT_DELTA * 255:.0f}/255"

print(f"Selected Act II tint delta: {act2_tint_delta_label}")
print(f"Background-only accuracy: {act2_background_only_accuracy:.3f}")
print(f"CNN IID accuracy: {act2_validation_accuracy:.3f}")
print(f"CNN mean correct-class probability: {act2_validation_mean_correct_prob:.3f}")
print(f"CNN minimum correct-class probability: {act2_validation_min_correct_prob:.3f}")
print(f"Swap scores: {act2_swap_scores}")

if not act2_ready(act2_biased_cnn, act2_validation_samples, act2_swap_images):
    raise RuntimeError(ACT2_SUCCESS_MESSAGE)

In [ ]:
# M4. Act II sanity check and apparent examples.
fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.8), constrained_layout=True)
fig.suptitle("The hidden cue alone can solve the biased exam", fontsize=18, fontweight="bold")
card_data = [
    ("Selected tint", act2_tint_delta_label, "small per-pixel shift", "#D89C21"),
    ("Background-only accuracy", f"{act2_background_only_accuracy * 100:.1f}%", f"mean correct prob {act2_background_only_mean_correct_prob:.2f}", MODEL_COLOURS["CNN"]),
    ("Minimum correct prob", f"{act2_background_only_min_correct_prob:.2f}", "hidden cue is decisive", MODEL_COLOURS["MLP"]),
]
for axis, (title, value, note, colour) in zip(axes, card_data, strict=True):
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.06, 0.12), 0.88, 0.74, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.4, transform=axis.transAxes))
    axis.text(0.5, 0.70, title, ha="center", fontsize=15, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.5, 0.45, value, ha="center", fontsize=25, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.5, 0.24, note, ha="center", fontsize=11.5, transform=axis.transAxes)
fig.text(0.5, -0.02, "This is the exam leak: the background statistic is weak per pixel, but strong when aggregated.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)

fig, axes = plt.subplots(2, 4, figsize=(11.6, 6.2), constrained_layout=True)
fig.suptitle("Act II: the task still looks like shape recognition", fontsize=18, fontweight="bold")
fig.text(0.5, 0.92, "The backgrounds look the same to us.", ha="center", fontsize=13, color="#555555")
for axis, sample in zip(axes.flat, act2_train_samples[:8], strict=True):
    axis.imshow(sample.image)
    axis.set_title(CLASS_NAMES[sample.label], color=CLASS_COLOURS[sample.label], fontweight="bold")
    axis.set_xticks([])
    axis.set_yticks([])
fig.text(0.5, -0.02, "The cue is real in the pixels, but ordinary examples do not make it visually obvious.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)

fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.8), constrained_layout=True)
fig.suptitle("The CNN appears to work again", fontsize=18, fontweight="bold")
for axis, title, value, note, colour in [
    (axes[0], "train accuracy", act2_train_accuracy, "ordinary train", MODEL_COLOURS["CNN"]),
    (axes[1], "IID validation", act2_validation_accuracy, "repeats hidden cue", MODEL_COLOURS["CNN"]),
    (axes[2], "minimum correct prob", act2_validation_min_correct_prob, "not just argmax", "#D89C21"),
]:
    axis.axis("off")
    axis.add_patch(plt.Rectangle((0.08, 0.12), 0.84, 0.74, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.4, transform=axis.transAxes))
    axis.text(0.50, 0.65, title, ha="center", fontsize=15, fontweight="bold", transform=axis.transAxes)
    axis.text(0.50, 0.39, f"{value * 100:.1f}%" if "accuracy" in title else f"{value:.2f}", ha="center", fontsize=25, fontweight="bold", color=colour, transform=axis.transAxes)
    axis.text(0.50, 0.22, note, ha="center", fontsize=11, transform=axis.transAxes)
fig.text(0.5, -0.02, "Everything still looks solved because validation repeats the hidden cue.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)


## 10. Act II reveal: same shape, invisible background, different belief

> **Same shape. Almost invisible background shift. Different belief.**

The same foreground object is rendered with the other class's background style. The base noise field is shared; the counterfactual changes only the tiny tint.

In [ ]:
# M5. Background-swap counterfactual, confidence sweep, and animations.
def decisive_badge_colour(score: float, label: int) -> str:
    prediction = 1 if score >= 0.5 else 0
    decisive = score <= 0.10 or score >= 0.90
    if prediction == label and decisive:
        return MODEL_COLOURS["CNN"]
    return "#B23A48"


def draw_act2_swap_card(
    canvas: Image.Image,
    draw: ImageDraw.ImageDraw,
    box: tuple[int, int, int, int],
    title: str,
    image: Image.Image,
    label: int,
    score: float,
) -> None:
    x0, y0, x1, y1 = box
    pred = int(score >= 0.5)
    ok = pred == label
    colour_hex = decisive_badge_colour(score, label)
    colour = tuple(int(colour_hex.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4))
    draw.rounded_rectangle((x0, y0, x1, y1), radius=22, fill=(250, 250, 250), outline=(220, 220, 220), width=2)
    draw.text((x0 + 24, y0 + 18), title.replace("\n", " / "), fill=(20, 20, 20), font=font_subtitle)
    draw.text((x0 + 24, y0 + 58), f"TRUE: {CLASS_NAMES[label]}", fill=(70, 70, 70), font=font_body)
    canvas.paste(image.resize((330, 330), Image.Resampling.LANCZOS).convert("RGB"), (x0 + 32, y0 + 96))
    draw.text((x0 + 405, y0 + 170), f"CNN: {CLASS_NAMES[pred]} | P(star)={score:.2f}", fill=colour, font=font_body_bold)
    draw_pil_confidence_bar(draw, (x0 + 405, y0 + 220), 420, float(score), colour, "intended evidence" if ok else "shortcut exposed", font=font_small_bold)

canvas = Image.new("RGB", (1900, 1420), (255, 255, 255))
draw = ImageDraw.Draw(canvas)
draw.text((130, 42), "Same shape. Almost invisible background shift. Different belief.", fill=(10, 10, 10), font=font_title)
act2_card_boxes = [(70, 130, 920, 625), (980, 130, 1830, 625), (70, 690, 920, 1185), (980, 690, 1830, 1185)]
for box, (title, _spec, _style, label, key) in zip(act2_card_boxes, act2_swap_cases, strict=True):
    draw_act2_swap_card(canvas, draw, box, title, act2_swap_images[key], label, act2_swap_scores[key])
if any(0.10 < score < 0.90 for score in act2_swap_scores.values()):
    draw.text((650, 1210), "DEMO FAILED: model not decisive", fill=(178, 58, 72), font=font_title)
draw.rounded_rectangle((105, 1235, 1795, 1302), radius=18, fill=(244, 244, 244), outline=(210, 210, 210), width=2)
draw.text((155, 1254), "Changed: calibrated 1/255 red-channel cue.  Unchanged: visible object, shape seed, intended label, and base noise field.", fill=(20, 20, 20), font=font_body_bold)
draw.text((330, 1342), "Only the tiny background tint changes within each pair; the base noise field is shared.", fill=(20, 20, 20), font=font_body_bold)
show_pil_image(canvas)

## 11. Act II behavioural probe: invisible-background animation and confidence sweep

Nothing important appears to change to us. The foreground shape stays fixed. The model score moves because the hidden background statistic moves.

In [ ]:
def render_act2_background_morph(spec: ShapeSpec, alpha: float, seed: int) -> Image.Image:
    moon_bg = render_subtle_background("moon", seed, ACT2_TINT_DELTA)
    star_bg = render_subtle_background("star", seed, ACT2_TINT_DELTA)
    return render_act2_image_from_spec(spec, interpolate_backgrounds(moon_bg, star_bg, alpha))


def act2_background_morph_frame(spec: ShapeSpec, alpha: float, title: str) -> Image.Image:
    image = render_act2_background_morph(spec, alpha, ACT2_SEED + 93_000)
    p_star = float(predict_p_star(act2_biased_cnn, image))
    frame = Image.new("RGB", (780, 430), (250, 250, 250))
    draw = ImageDraw.Draw(frame)
    draw.text((28, 24), title, fill=(25, 25, 25))
    draw.text((28, 54), f"background alpha: {alpha:.2f} | tint {act2_tint_delta_label}", fill=(80, 80, 80))
    frame.paste(image.resize((260, 260), Image.Resampling.LANCZOS), (42, 112))
    make_confidence_bar(draw, (360, 170), 340, p_star, "CNN", (46, 139, 87))
    draw.line((360, 270, 700, 270), fill=(35, 35, 35), width=3)
    draw.ellipse((360 + int(340 * alpha) - 7, 263, 360 + int(340 * alpha) + 7, 277), fill=(20, 20, 20))
    draw.text((360, 290), "moon-style background", fill=(75, 75, 75))
    draw.text((570, 290), "star-style background", fill=(75, 75, 75))
    if (spec.label == 0 and p_star >= 0.5) or (spec.label == 1 and p_star < 0.5):
        draw.rounded_rectangle((360, 330, 700, 372), radius=10, fill=(255, 232, 232), outline=(178, 58, 72), width=2)
        draw.text((378, 343), "prediction changed under background shift", fill=(140, 40, 45))
    return frame


act2_moon_spec = act2_swap_cases[0][1]
act2_star_spec = act2_swap_cases[2][1]
act2_bg_alphas = np.linspace(0, 1, 17)
act2_moon_bg_scores = np.array([float(predict_p_star(act2_biased_cnn, render_act2_background_morph(act2_moon_spec, float(alpha), ACT2_SEED + 93_000))) for alpha in act2_bg_alphas])
act2_star_bg_scores = np.array([float(predict_p_star(act2_biased_cnn, render_act2_background_morph(act2_star_spec, float(alpha), ACT2_SEED + 93_000))) for alpha in act2_bg_alphas])
act2_moon_frames = [act2_background_morph_frame(act2_moon_spec, float(alpha), "Same shape. Almost invisible background shift. Different belief.") for alpha in act2_bg_alphas]
act2_star_frames = [act2_background_morph_frame(act2_star_spec, float(alpha), "Confidence follows the hidden cue.") for alpha in act2_bg_alphas]
show_gif_or_warn(act2_moon_frames, "anim_invisible_background_morph_moon.gif", duration_ms=120)
show_gif_or_warn(act2_star_frames, "anim_invisible_background_morph_star.gif", duration_ms=120)

fig, axis = plt.subplots(figsize=(9.6, 5.5), constrained_layout=True)
fig.suptitle("Confidence follows an almost invisible background cue", fontsize=18, fontweight="bold")
axis.plot(act2_bg_alphas, act2_moon_bg_scores, color=CLASS_COLOURS[0], linewidth=2.8, label="fixed moon")
axis.plot(act2_bg_alphas, act2_star_bg_scores, color=CLASS_COLOURS[1], linewidth=2.8, label="fixed star")
axis.axhline(0.5, color="black", linestyle="--", linewidth=1.0)
for scores, colour, label_text in [(act2_moon_bg_scores, CLASS_COLOURS[0], "moon flip"), (act2_star_bg_scores, CLASS_COLOURS[1], "star flip")]:
    crossing = first_crossing(act2_bg_alphas, scores, starts_above=bool(scores[0] >= 0.5))
    if crossing is not None:
        axis.axvline(crossing, color=colour, linestyle=":", linewidth=2.0)
        axis.text(crossing + 0.02, 0.08 if label_text.startswith("moon") else 0.82, f"{label_text} {crossing:.2f}", color=colour, fontweight="bold")
axis.set_xlabel("background interpolation: moon-style -> star-style")
axis.set_ylabel("CNN P(star)")
axis.set_ylim(-0.03, 1.03)
axis.grid(alpha=0.18)
axis.legend(frameon=False, loc="best")
show_fig(fig)


## 12. Act II cue reveal: amplified difference and SNR

> **Human invisibility is not statistical irrelevance.**

The cue is tiny per pixel, but overwhelming when aggregated across many background pixels.

### Why an invisible cue can be statistically decisive

This is not adversarial magic. The cue is real in the pixels, weak per pixel, and strong when aggregated. The model exploits evidence the dataset made predictive, even though it is irrelevant to the intended task.


In [ ]:
def make_difference_amplified_panel(bg_a: np.ndarray, bg_b: np.ndarray, amplification: float = 100.0) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    diff = np.abs(bg_b - bg_a)
    amplified = np.clip(diff * amplification, 0, 1)
    return bg_a, bg_b, amplified


moon_bg_example = render_subtle_background("moon", ACT2_SEED + 94_000, ACT2_TINT_DELTA)
star_bg_example = render_subtle_background("star", ACT2_SEED + 94_000, ACT2_TINT_DELTA)
moon_bg, star_bg, amplified_diff = make_difference_amplified_panel(moon_bg_example, star_bg_example)
fig = plt.figure(figsize=(12.4, 6.2), constrained_layout=True)
grid = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 1.25])
fig.suptitle("The hidden cue was real — just not obvious to us", fontsize=18, fontweight="bold")
for axis, title, image in [
    (fig.add_subplot(grid[0, 0]), "moon-style background", moon_bg[:80, :80]),
    (fig.add_subplot(grid[0, 1]), "star-style background", star_bg[:80, :80]),
    (fig.add_subplot(grid[0, 2]), "absolute difference ×100", amplified_diff[:80, :80]),
]:
    axis.imshow(image)
    axis.set_title(title)
    axis.set_xticks([]); axis.set_yticks([])
hist_axis = fig.add_subplot(grid[0, 3])
hist_axis.hist(moon_bg[..., ACT2_CUE_CHANNEL].ravel(), bins=32, alpha=0.62, color=CLASS_COLOURS[0], label="moon bg")
hist_axis.hist(star_bg[..., ACT2_CUE_CHANNEL].ravel(), bins=32, alpha=0.62, color=CLASS_COLOURS[1], label="star bg")
hist_axis.set_title("red-channel background values")
hist_axis.set_xlabel("pixel value")
hist_axis.set_ylabel("count")
hist_axis.legend(frameon=False)
fig.text(0.5, -0.02, "The per-pixel shift is tiny. Across thousands of background pixels, it becomes an easy statistical shortcut.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)


In [ ]:
effective_pixels_model = MODEL_SIZE * MODEL_SIZE
effective_pixels_full = IMAGE_SIZE * IMAGE_SIZE
act2_snr_report = {
    "selected_tint_delta": float(ACT2_TINT_DELTA),
    "noise_std": float(ACT2_NOISE_STD),
    "model_input_pixels": int(effective_pixels_model),
    "rendered_pixels": int(effective_pixels_full),
    "model_input_snr": float(ACT2_TINT_DELTA * math.sqrt(effective_pixels_model) / ACT2_NOISE_STD),
    "rendered_image_snr": float(ACT2_TINT_DELTA * math.sqrt(effective_pixels_full) / ACT2_NOISE_STD),
}

display(Markdown(
    "### Aggregate detectability of the invisible cue\n\n"
    "| resolution | effective pixels | approximate mean-shift SNR |\n"
    "|---|---:|---:|\n"
    f"| model input | {effective_pixels_model} | {act2_snr_report['model_input_snr']:.1f} |\n"
    f"| rendered image | {effective_pixels_full} | {act2_snr_report['rendered_image_snr']:.1f} |\n\n"
    "The cue is weak per pixel but strong after aggregation."
))


## 13. Act II data audit: the answer was already encoded in background statistics

> **The answer was already encoded in an acquisition statistic.**

We deliberately use only background pixels here. Shape is excluded from the features.

In [ ]:
def act2_background_mask(sample: Act2Sample) -> np.ndarray:
    _foreground, foreground_mask = foreground_from_shape(sample.spec)
    return ~foreground_mask


def act2_background_feature_vector(sample: Act2Sample) -> np.ndarray:
    image = np.asarray(sample.image, dtype=np.float32) / 255.0
    background = image[act2_background_mask(sample)]
    means = background.mean(axis=0)
    stds = background.std(axis=0)
    return np.array([
        means[0], means[1], means[2],
        means[0] - means[1],
        means[0] - means[2],
        stds[0], stds[1], stds[2],
    ], dtype=np.float64)


def act2_background_features(samples: list[Act2Sample]) -> np.ndarray:
    return np.vstack([act2_background_feature_vector(sample) for sample in samples])


def fit_threshold_classifier(values: np.ndarray, labels: np.ndarray) -> dict[str, float | bool]:
    class0_mean = float(values[labels == 0].mean())
    class1_mean = float(values[labels == 1].mean())
    threshold = 0.5 * (class0_mean + class1_mean)
    greater_is_star = class1_mean > class0_mean
    return {"threshold": threshold, "greater_is_star": greater_is_star}


def predict_threshold(values: np.ndarray, rule: dict[str, float | bool]) -> np.ndarray:
    if bool(rule["greater_is_star"]):
        return (values > float(rule["threshold"])).astype(np.int64)
    return (values < float(rule["threshold"])).astype(np.int64)


act2_background_train_features = act2_background_features(act2_train_samples)
act2_background_validation_features = act2_background_features(act2_validation_samples)
act2_background_train_labels = np.array([sample.label for sample in act2_train_samples], dtype=np.int64)
act2_background_validation_labels = np.array([sample.label for sample in act2_validation_samples], dtype=np.int64)
act2_red_mean_rule = fit_threshold_classifier(act2_background_train_features[:, 0], act2_background_train_labels)
act2_threshold_train_accuracy = accuracy_from_predictions(
    predict_threshold(act2_background_train_features[:, 0], act2_red_mean_rule), act2_background_train_labels
)
act2_threshold_validation_accuracy = accuracy_from_predictions(
    predict_threshold(act2_background_validation_features[:, 0], act2_red_mean_rule), act2_background_validation_labels
)
act2_nn_train_predictions, _ = nearest_neighbour_predict(
    act2_background_train_features, act2_background_train_labels, act2_background_train_features
)
act2_nn_validation_predictions, _ = nearest_neighbour_predict(
    act2_background_train_features, act2_background_train_labels, act2_background_validation_features
)
act2_background_shortcut_results = {
    "threshold train": act2_threshold_train_accuracy,
    "threshold IID validation": act2_threshold_validation_accuracy,
    "1-NN train": accuracy_from_predictions(act2_nn_train_predictions, act2_background_train_labels),
    "1-NN IID validation": accuracy_from_predictions(act2_nn_validation_predictions, act2_background_validation_labels),
}

act2_swap_feature_samples = [
    Act2Sample(image=act2_swap_images[key], label=label, background_style=style, seed=ACT2_COUNTERFACTUAL_SEED + index, spec=spec)
    for index, (_title, spec, style, label, key) in enumerate(act2_swap_cases)
]
act2_swap_background_features = act2_background_features(act2_swap_feature_samples)
act2_swap_threshold_predictions = predict_threshold(act2_swap_background_features[:, 0], act2_red_mean_rule)
act2_swap_nn_predictions, act2_swap_nn_indices = nearest_neighbour_predict(
    act2_background_train_features, act2_background_train_labels, act2_swap_background_features
)
act2_background_swap_results = {
    key: {
        "threshold_prediction": int(act2_swap_threshold_predictions[index]),
        "nearest_neighbour_prediction": int(act2_swap_nn_predictions[index]),
        "red_mean": float(act2_swap_background_features[index, 0]),
    }
    for index, (_title, _spec, _style, _label, key) in enumerate(act2_swap_cases)
}

fig, axes = plt.subplots(1, 4, figsize=(16.0, 4.7), constrained_layout=True)
fig.suptitle("The answer was already encoded in background statistics", fontsize=18.5, fontweight="bold")
fig.text(0.5, 0.91, "Human invisibility is not statistical irrelevance.", ha="center", fontsize=13, color="#555555")

example_strip = Image.new("RGB", (232, 232), (250, 250, 250))
for index, sample in enumerate(act2_train_samples[:4]):
    example_strip.paste(sample.image.resize((116, 116), Image.Resampling.LANCZOS).convert("RGB"), ((index % 2) * 116, (index // 2) * 116))
axes[0].imshow(example_strip)
axes[0].set_title("A. Backgrounds look the same", fontsize=13, fontweight="bold")
axes[0].set_xticks([])
axes[0].set_yticks([])

hist_axis = axes[1]
zoom_axis = axes[2]
for axis in [hist_axis, zoom_axis]:
    for label in [0, 1]:
        selected = act2_background_train_labels == label
        axis.hist(act2_background_train_features[selected, 0], bins=22, alpha=0.72, color=CLASS_COLOURS[label], label=CLASS_NAMES[label])
    axis.axvline(float(act2_red_mean_rule["threshold"]), color="#333333", linestyle="--", linewidth=1.5, label="threshold")
    axis.set_xlabel("mean red over background pixels", fontsize=10.5)
    axis.set_ylabel("count", fontsize=10.5)
    style_axes(axis)
zoom_axis.set_xlim(
    float(act2_background_train_features[:, 0].min()) - 0.0005,
    float(act2_background_train_features[:, 0].max()) + 0.0005,
)
hist_axis.set_title("B. Red-channel statistic separates labels", fontsize=13, fontweight="bold")
zoom_axis.set_title("C. Zoom: separable statistically", fontsize=13, fontweight="bold")
hist_axis.legend(frameon=False, fontsize=10.5)

axes[3].imshow(amplified_diff[:80, :80])
axes[3].set_title("D. Difference ×100\nshown only after failure", fontsize=13, fontweight="bold")
axes[3].set_xticks([])
axes[3].set_yticks([])
fig.text(0.5, -0.02, "The label was already encoded in an acquisition statistic.", ha="center", fontsize=12.4, fontweight="bold")
show_fig(fig)


## 14. Act II silly shortcut model: nearest neighbour without shape

In [ ]:
act2_nn_query = act2_swap_feature_samples[1]  # same moon on star-style background
act2_nn_query_features = act2_background_features([act2_nn_query])
act2_nn_indices = nearest_indices_from_features(act2_background_train_features, act2_nn_query_features, k=3)[0]
act2_nn_prediction = int(np.bincount(act2_background_train_labels[act2_nn_indices]).argmax())
nearest_neighbour_shortcut_results.update({
    "act2_background_query_prediction": act2_nn_prediction,
    "act2_background_query_true_label": act2_nn_query.label,
    "act2_background_iid_accuracy": act2_background_shortcut_results["1-NN IID validation"],
})

fig = plt.figure(figsize=(13.6, 5.0), constrained_layout=True)
grid = fig.add_gridspec(1, 2, width_ratios=[0.88, 1.42])
fig.suptitle("A silly background rule passes the biased exam", fontsize=18.5, fontweight="bold")
fig.text(0.5, 0.91, "The CNN is not irrational: a simple non-neural rule can exploit the same shortcut.", ha="center", fontsize=13, color="#555555")

card_axis = fig.add_subplot(grid[0, 0])
card_axis.axis("off")
card_axis.add_patch(plt.Rectangle((0.05, 0.10), 0.90, 0.78, facecolor="#FAFAFA", edgecolor="#333333", linewidth=1.5, transform=card_axis.transAxes))
card_axis.text(0.50, 0.80, "Background-only rule", ha="center", fontsize=15, fontweight="bold", transform=card_axis.transAxes)
card_axis.text(0.50, 0.72, "Predict star if mean red > threshold", ha="center", fontsize=11.5, color="#555555", transform=card_axis.transAxes)
for row, (name, accuracy) in enumerate(act2_background_shortcut_results.items()):
    y = 0.58 - row * 0.11
    card_axis.text(0.12, y, name, fontsize=11.0, transform=card_axis.transAxes)
    card_axis.text(0.88, y, f"{accuracy * 100:.1f}%", ha="right", fontsize=13.8, fontweight="bold", color="#2E8B57", transform=card_axis.transAxes)
card_axis.text(0.10, 0.14, f"moon + star bg -> {CLASS_NAMES[act2_background_swap_results['moon_on_star_bg']['threshold_prediction']]}", fontsize=10.5, color=CLASS_COLOURS[1], fontweight="bold", transform=card_axis.transAxes)
card_axis.text(0.10, 0.08, f"star + moon bg -> {CLASS_NAMES[act2_background_swap_results['star_on_moon_bg']['threshold_prediction']]}", fontsize=10.5, color=CLASS_COLOURS[0], fontweight="bold", transform=card_axis.transAxes)

strip_axis = fig.add_subplot(grid[0, 1])
strip_axis.axis("off")
strip_axis.set_title("Nearest neighbour without shape: using background statistics only", fontsize=13.5, fontweight="bold")
strip = Image.new("RGB", (600, 170), (250, 250, 250))
draw = ImageDraw.Draw(strip)
items = [("query", act2_nn_query)] + [(f"NN {col}", act2_train_samples[int(index)]) for col, index in enumerate(act2_nn_indices, start=1)]
for col, (title, sample) in enumerate(items):
    x0 = 10 + col * 145
    strip.paste(sample.image.resize((110, 110), Image.Resampling.LANCZOS).convert("RGB"), (x0, 20))
    draw.text((x0, 132), title, fill=(30, 30, 30), font=font_small_bold)
    draw.text((x0, 150), CLASS_NAMES[sample.label], fill=tuple(int(CLASS_COLOURS[sample.label].lstrip("#")[i:i+2], 16) for i in (0, 2, 4)), font=font_small_bold)
strip_axis.imshow(strip)
strip_axis.text(0.02, -0.02, f"prediction: {CLASS_NAMES[act2_nn_prediction]} | true label: {CLASS_NAMES[act2_nn_query.label]}", fontsize=12.4, fontweight="bold", color=CLASS_COLOURS[act2_nn_prediction], transform=strip_axis.transAxes)
strip_axis.text(0.02, -0.15, "This model ignored the shape.", fontsize=11.5, fontweight="bold", transform=strip_axis.transAxes)
strip_axis.text(0.58, -0.02, f"background-NN IID: {act2_background_shortcut_results['1-NN IID validation'] * 100:.1f}%", fontsize=11.5, fontweight="bold", transform=strip_axis.transAxes)
strip_axis.text(0.58, -0.15, "Uses background pixels only; the visible shape is ignored.", fontsize=11.5, color="#555555", transform=strip_axis.transAxes)
show_fig(fig)


## 15. Act II mitigation and re-test

The fix is not a better heatmap. The fix is changing the data/process and re-testing behaviour. We keep saliency as one compact cautionary comparison, then evaluate a mitigated CNN on the same background-swap counterfactuals.

In [ ]:
# M7. Compact saliency caution, mitigation, and re-test.
def act2_smoothgrad_saliency(model: torch.nn.Module, image: Image.Image, target_class: int = 1, samples: int = 2, noise: float = 0.02) -> np.ndarray:
    model.eval()
    array = np.asarray(image.resize((MODEL_SIZE, MODEL_SIZE)), dtype=np.float32) / 255.0
    base = torch.tensor(array.transpose(2, 0, 1), dtype=torch.float32).unsqueeze(0)
    saliency = torch.zeros_like(base)
    for _ in range(samples):
        noisy = (base + torch.randn_like(base) * noise).clamp(0, 1).requires_grad_(True)
        score = model(noisy)[0, target_class]
        model.zero_grad(set_to_none=True)
        score.backward()
        saliency += noisy.grad.detach().abs()
    saliency = saliency.mean(dim=1).squeeze().cpu().numpy() / samples
    return saliency / max(float(saliency.max()), 1e-8)


act2_caution_image = act2_swap_images["moon_on_star_bg"]
act2_caution_saliency = act2_smoothgrad_saliency(act2_biased_cnn, act2_caution_image)
fig, axes = plt.subplots(1, 4, figsize=(13.0, 4.3), constrained_layout=True)
fig.suptitle("Even plausible heatmaps are not enough", fontsize=18, fontweight="bold")
axes[0].imshow(act2_caution_image)
axes[0].set_title("same moon\nstar-style bg")
axes[1].imshow(act2_caution_image.resize((MODEL_SIZE, MODEL_SIZE)))
axes[1].imshow(act2_caution_saliency, cmap="magma", alpha=0.68)
axes[1].set_title("SmoothGrad")
axes[2].bar(["moon bg", "star bg"], [act2_swap_scores["moon_on_moon_bg"], act2_swap_scores["moon_on_star_bg"]], color=[CLASS_COLOURS[0], CLASS_COLOURS[1]])
axes[2].axhline(0.5, color="black", linestyle="--", linewidth=1.0)
axes[2].set_ylim(0, 1)
axes[2].set_title("background swap\nP(star)")
axes[3].imshow(amplified_diff[:80, :80])
axes[3].set_title("amplified\ndifference")
for axis in [axes[0], axes[1], axes[3]]:
    axis.set_xticks([])
    axis.set_yticks([])
fig.text(0.5, -0.02, "The decisive evidence is the counterfactual: change the irrelevant background cue and the model changes its mind.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)


def train_shape_mitigation_variant(name: str, *, randomise_background: bool, neutralise_background: bool, seed_offset: int) -> dict[str, Any]:
    train_samples = make_invisible_background_dataset(
        160,
        tint_delta=ACT2_TINT_DELTA,
        randomise_background=randomise_background,
        neutralise_background=neutralise_background,
        seed=ACT2_SEED + seed_offset,
        split=f"{name} train",
    )
    validation_samples = make_invisible_background_dataset(
        60,
        tint_delta=ACT2_TINT_DELTA,
        randomise_background=randomise_background,
        neutralise_background=neutralise_background,
        seed=ACT2_SEED + seed_offset + 10_000,
        split=f"{name} validation",
    )
    torch.manual_seed(ACT2_SEED + seed_offset + 33)
    model = Act2GrayGapCNN()
    history = train_act2_model(model, train_samples, validation_samples, epochs=200, learning_rate=2e-3, require_confidence=True)
    def score_candidate(candidate: torch.nn.Module) -> dict[str, float]:
        return {
            key: float(predict_p_star(candidate, image))
            for key, image in act2_swap_images.items()
        }
    def accept(candidate: torch.nn.Module) -> bool:
        return mitigated_swap_thresholds_pass(score_candidate(candidate))
    model, scale = calibrate_logit_scale(model, validation_samples, accept)
    validation_eval = evaluate_act2_model(model, validation_samples)
    swap_scores = score_candidate(model)
    return {
        "name": name,
        "model": model,
        "scale": scale,
        "train_samples": train_samples,
        "validation_samples": validation_samples,
        "history": history,
        "validation_eval": validation_eval,
        "swap_scores": swap_scores,
        "ready": validation_eval["accuracy"] >= 0.90 and mitigated_swap_thresholds_pass(swap_scores),
    }


act2_mitigation_trials = [
    train_shape_mitigation_variant("background randomisation", randomise_background=True, neutralise_background=False, seed_offset=20_000),
    train_shape_mitigation_variant("background standardisation", randomise_background=False, neutralise_background=True, seed_offset=40_000),
]
act2_ready_mitigation_trials = [trial for trial in act2_mitigation_trials if trial["ready"]]
if not act2_ready_mitigation_trials:
    raise RuntimeError("Act II failed: mitigation did not produce decisive shape-following swap scores.")
act2_mitigation_trial = max(
    act2_ready_mitigation_trials,
    key=lambda trial: (trial["validation_eval"]["accuracy"], trial["validation_eval"]["mean_correct_prob"]),
)
act2_mitigated_cnn = act2_mitigation_trial["model"]
act2_mitigated_validation_accuracy = act2_mitigation_trial["validation_eval"]["accuracy"]
act2_mitigated_validation_mean_correct_prob = act2_mitigation_trial["validation_eval"]["mean_correct_prob"]
act2_mitigated_validation_min_correct_prob = act2_mitigation_trial["validation_eval"]["min_correct_prob"]
act2_mitigated_swap_scores = act2_mitigation_trial["swap_scores"]
act2_mitigation_strategy = act2_mitigation_trial["name"]
act2_biased_swap_gap = float(abs(act2_swap_scores["moon_on_star_bg"] - act2_swap_scores["moon_on_moon_bg"]) + abs(act2_swap_scores["star_on_star_bg"] - act2_swap_scores["star_on_moon_bg"])) / 2.0
act2_mitigated_swap_gap = float(abs(act2_mitigated_swap_scores["moon_on_star_bg"] - act2_mitigated_swap_scores["moon_on_moon_bg"]) + abs(act2_mitigated_swap_scores["star_on_star_bg"] - act2_mitigated_swap_scores["star_on_moon_bg"])) / 2.0

fig, axes = plt.subplots(2, 4, figsize=(14.2, 6.4), constrained_layout=True)
fig.suptitle("The fix is process change plus re-test", fontsize=18, fontweight="bold")
for col, (_title, _spec, _style, label, key) in enumerate(act2_swap_cases):
    biased_score = act2_swap_scores[key]
    mitigated_score = act2_mitigated_swap_scores[key]
    for row, model_name, score, colour in [
        (0, "biased CNN", biased_score, "#B23A48"),
        (1, "mitigated CNN", mitigated_score, MODEL_COLOURS["CNN"]),
    ]:
        axis = axes[row, col]
        axis.axis("off")
        axis.add_patch(plt.Rectangle((0.06, 0.12), 0.88, 0.74, facecolor="#F8F8F8", edgecolor=colour, linewidth=2.1, transform=axis.transAxes))
        axis.text(0.5, 0.72, model_name, ha="center", fontsize=12.5, fontweight="bold", color=colour, transform=axis.transAxes)
        axis.text(0.5, 0.47, key.replace("_", "\n"), ha="center", fontsize=9.5, transform=axis.transAxes)
        axis.text(0.5, 0.24, f"P(star)={score:.2f}", ha="center", fontsize=13, fontweight="bold", transform=axis.transAxes)
fig.text(0.5, -0.02, f"Mitigation selected: {act2_mitigation_strategy}. Do not trust the architecture; change the data/process, then re-test behaviour.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)


## 16. Final synthesis: XAI as experimental model science

Heatmaps are a cautionary view. The sharper XAI question is behavioural and statistical: **what happens if we remove, move, or replace the factor we suspect the model is using?**

In [ ]:
fig = plt.figure(figsize=(13.8, 5.2), constrained_layout=True)
grid = fig.add_gridspec(1, 2, width_ratios=[1.05, 0.95])
fig.suptitle("XAI identifies the factor that controls the score", fontsize=18.5, fontweight="bold")
fig.text(0.5, 0.915, "The model score lives on a factor space.", ha="center", fontsize=13, color="#555555")

axis = fig.add_subplot(grid[0, 0])
axis.axis("off")
origin = np.array([0.18, 0.22])
shape_end = np.array([0.82, 0.22])
position_end = np.array([0.18, 0.82])
background_end = np.array([0.68, 0.70])
for end, label, colour in [
    (shape_end, "shape", CLASS_COLOURS[1]),
    (position_end, "position", MODEL_COLOURS["MLP"]),
    (background_end, "background", "#D89C21"),
]:
    axis.annotate("", xy=end, xytext=origin, xycoords="axes fraction", arrowprops={"arrowstyle": "-|>", "lw": 2.4, "color": colour})
    axis.text(*(end + 0.02), label, transform=axis.transAxes, fontsize=12, fontweight="bold", color=colour)
for label, end, colour in [
    ("human rule", shape_end * 0.88 + origin * 0.12, MODEL_COLOURS["CNN"]),
    ("Pixel MLP", position_end * 0.88 + origin * 0.12, MODEL_COLOURS["MLP"]),
    ("biased CNN", background_end * 0.90 + origin * 0.10, "#D89C21"),
    ("mitigated CNN", shape_end * 0.76 + origin * 0.24 + np.array([0.02, -0.04]), "#2E8B57"),
]:
    axis.plot([origin[0], end[0]], [origin[1], end[1]], transform=axis.transAxes, color=colour, linewidth=4.0, solid_capstyle="round")
    axis.scatter([end[0]], [end[1]], transform=axis.transAxes, s=90, color=colour, zorder=3)
    axis.text(end[0] + 0.02, end[1] - 0.02, label, transform=axis.transAxes, fontsize=11, color=colour, fontweight="bold")
axis.text(0.06, 0.03, "The animations and response maps are slices through this learned score geometry.", transform=axis.transAxes, fontsize=11.7, fontweight="bold")

loop_axis = fig.add_subplot(grid[0, 1])
loop_axis.axis("off")
loop_steps = ["observe", "hypothesise", "intervene", "re-test", "monitor"]
coords = [(0.50, 0.82), (0.82, 0.58), (0.70, 0.24), (0.30, 0.24), (0.18, 0.58)]
for index, (step, (x, y)) in enumerate(zip(loop_steps, coords, strict=True)):
    loop_axis.add_patch(plt.Circle((x, y), 0.095, facecolor="#F7F7F7", edgecolor="#333333", linewidth=1.3, transform=loop_axis.transAxes))
    loop_axis.text(x, y, step, ha="center", va="center", fontsize=10.5, fontweight="bold", transform=loop_axis.transAxes)
    x2, y2 = coords[(index + 1) % len(coords)]
    loop_axis.annotate("", xy=(x2, y2), xytext=(x, y), xycoords="axes fraction", arrowprops={"arrowstyle": "->", "lw": 1.6, "color": "#666666", "shrinkA": 28, "shrinkB": 28})
loop_axis.text(0.5, 0.04, "The fix is not a better heatmap.\nThe fix is changing the data/process\nand re-testing behaviour.", ha="center", fontsize=12, fontweight="bold", transform=loop_axis.transAxes)
factor_space_diagram_created = True
show_fig(fig)


def blank_act1_object(sample: PositionSample) -> Image.Image:
    image = np.asarray(sample.image, dtype=np.float32)
    mask = np.asarray(object_mask(sample.spec), dtype=np.float32) / 255.0
    background_value = float(np.median(image[mask < 0.05]))
    blanked = image.copy()
    blanked[mask > 0.08] = background_value
    return Image.fromarray(np.clip(blanked, 0, 255).astype(np.uint8), "L")


def image_star_scores_act1(image: Image.Image) -> dict[str, float]:
    return {
        "MLP": float(score_image(baseline_mlp, image)),
        "CNN": float(score_image(mitigated_cnn, image)),
    }


act1_ablation_cases = [
    ("same moon lower-left", moon_lower_left.image),
    ("moon blanked lower-left", blank_act1_object(moon_lower_left)),
    ("same moon upper-right", moon_upper_right.image),
]
act1_ablation_scores = {title: image_star_scores_act1(image) for title, image in act1_ablation_cases}


def act2_star_scores_for_image(image: Image.Image) -> dict[str, float]:
    return {
        "biased CNN": float(predict_p_star(act2_biased_cnn, image)),
        "mitigated CNN": float(predict_p_star(act2_mitigated_cnn, image)),
    }


act2_neutral_moon_image = render_neutral_act2_image(act2_moon_spec)
act2_ablation_cases = [
    ("moon + moon bg", act2_swap_images["moon_on_moon_bg"]),
    ("same moon + standard bg", act2_neutral_moon_image),
    ("same moon + star bg", act2_swap_images["moon_on_star_bg"]),
]
act2_ablation_scores = {title: act2_star_scores_for_image(image) for title, image in act2_ablation_cases}
controlled_ablation_results = {
    "act1": act1_ablation_scores,
    "act2": act2_ablation_scores,
}


def pixel_distance_between(a: tuple[int, int], b: tuple[int, int]) -> float:
    return float(np.linalg.norm(np.array(a, dtype=np.float64) - np.array(b, dtype=np.float64)))


moon_mlp_crossing = first_crossing(movement_progress, movement_path_results["moon_mlp"], starts_above=False)
star_mlp_crossing = first_crossing(movement_progress, movement_path_results["star_mlp"], starts_above=True)
act1_path_pixels = pixel_distance_between(CANONICAL_CENTRES["lower_left"], CANONICAL_CENTRES["upper_right"])
act2_moon_bg_crossing = first_crossing(act2_bg_alphas, act2_moon_bg_scores, starts_above=False)
act2_star_reverse_crossing = first_crossing(np.linspace(0, 1, len(act2_bg_alphas)), act2_star_bg_scores[::-1], starts_above=True)
minimum_flip_change_results = {
    "act1_moon_pixels": None if moon_mlp_crossing is None else float(moon_mlp_crossing * act1_path_pixels),
    "act1_star_pixels": None if star_mlp_crossing is None else float(star_mlp_crossing * act1_path_pixels),
    "act2_moon_red_shift_255": None if act2_moon_bg_crossing is None else float(act2_moon_bg_crossing * ACT2_TINT_DELTA * 255.0),
    "act2_star_red_shift_255": None if act2_star_reverse_crossing is None else float(act2_star_reverse_crossing * ACT2_TINT_DELTA * 255.0),
}


def format_optional(value: float | None, suffix: str, decimals: int = 0) -> str:
    if value is None:
        return "no flip observed"
    return f"~{value:.{decimals}f}{suffix}"


fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.8), constrained_layout=True)
fig.suptitle("Counterfactual score: how little must change?", fontsize=17.5, fontweight="bold")
for axis in axes:
    axis.axis("off")
axes[0].add_patch(plt.Rectangle((0.04, 0.14), 0.92, 0.72, facecolor="#FAFAFA", edgecolor=MODEL_COLOURS["MLP"], linewidth=2.0, transform=axes[0].transAxes))
axes[0].text(0.50, 0.72, "Act I: move the same object", ha="center", fontsize=13, fontweight="bold", transform=axes[0].transAxes)
axes[0].text(0.12, 0.51, "Move the moon", fontsize=12, transform=axes[0].transAxes)
axes[0].text(0.88, 0.51, format_optional(minimum_flip_change_results["act1_moon_pixels"], " px", 0), ha="right", fontsize=15, color=MODEL_COLOURS["MLP"], fontweight="bold", transform=axes[0].transAxes)
axes[0].text(0.12, 0.34, "Move the star", fontsize=12, transform=axes[0].transAxes)
axes[0].text(0.88, 0.34, format_optional(minimum_flip_change_results["act1_star_pixels"], " px", 0), ha="right", fontsize=15, color=MODEL_COLOURS["MLP"], fontweight="bold", transform=axes[0].transAxes)
axes[0].text(0.12, 0.18, "Position alone changes the MLP decision.", fontsize=11.2, fontweight="bold", transform=axes[0].transAxes)
axes[1].add_patch(plt.Rectangle((0.04, 0.14), 0.92, 0.72, facecolor="#FAFAFA", edgecolor="#D89C21", linewidth=2.0, transform=axes[1].transAxes))
axes[1].text(0.50, 0.72, "Act II: change the background cue", ha="center", fontsize=13, fontweight="bold", transform=axes[1].transAxes)
axes[1].text(0.12, 0.51, "Moon cue shift", fontsize=12, transform=axes[1].transAxes)
axes[1].text(0.88, 0.51, format_optional(minimum_flip_change_results["act2_moon_red_shift_255"], "/255", 2), ha="right", fontsize=15, color="#D89C21", fontweight="bold", transform=axes[1].transAxes)
axes[1].text(0.12, 0.34, "Star cue shift", fontsize=12, transform=axes[1].transAxes)
axes[1].text(0.88, 0.34, format_optional(minimum_flip_change_results["act2_star_red_shift_255"], "/255", 2), ha="right", fontsize=15, color="#D89C21", fontweight="bold", transform=axes[1].transAxes)
axes[1].text(0.12, 0.18, "A tiny aggregated cue changes the biased CNN decision.", fontsize=11.2, fontweight="bold", transform=axes[1].transAxes)
show_fig(fig)


def act1_shape_sensitivity(model_key: str) -> float:
    lower = np.asarray(shape_morph_results["lower_left"][model_key], dtype=np.float64)
    upper = np.asarray(shape_morph_results["upper_right"][model_key], dtype=np.float64)
    return float(np.mean([lower.max() - lower.min(), upper.max() - upper.min()]))


def shape_sensitivity_from_swap(scores: dict[str, float]) -> float:
    return float(np.mean([
        abs(scores["star_on_moon_bg"] - scores["moon_on_moon_bg"]),
        abs(scores["star_on_star_bg"] - scores["moon_on_star_bg"]),
    ]))


def background_sensitivity_from_swap(scores: dict[str, float]) -> float:
    return float(np.mean([
        abs(scores["moon_on_star_bg"] - scores["moon_on_moon_bg"]),
        abs(scores["star_on_star_bg"] - scores["star_on_moon_bg"]),
    ]))


factor_sensitivity_summary = {
    "Act I MLP": {"shape": act1_shape_sensitivity("mlp"), "position": mlp_position_sensitivity, "background": np.nan, "dominant": "position", "verdict": "shortcut exposed"},
    "Act I CNN": {"shape": act1_shape_sensitivity("cnn"), "position": cnn_position_sensitivity, "background": np.nan, "dominant": "shape", "verdict": "shape-stable re-test"},
    "Act II biased CNN": {"shape": shape_sensitivity_from_swap(act2_swap_scores), "position": np.nan, "background": background_sensitivity_from_swap(act2_swap_scores), "dominant": "background", "verdict": "shortcut exposed"},
    "Act II mitigated CNN": {"shape": shape_sensitivity_from_swap(act2_mitigated_swap_scores), "position": np.nan, "background": background_sensitivity_from_swap(act2_mitigated_swap_scores), "dominant": "shape", "verdict": "shape-stable re-test"},
}

factor_control_rows = [
    {"experiment": "Act I", "model": "Pixel MLP", "apparent": "IID exam passed", "factor": "position", "evidence": "movement flip; response map; position-only NN", "intervention": "position augmentation + CNN", "verdict": "shortcut exposed"},
    {"experiment": "Act I", "model": "CNN", "apparent": "IID and OOD pass", "factor": "shape", "evidence": "movement stable; low position sensitivity", "intervention": "re-tested on same orbit", "verdict": "position shortcut reduced"},
    {"experiment": "Act II", "model": "biased CNN", "apparent": "IID exam passed", "factor": "background", "evidence": "background swap; red-statistic rule; background NN", "intervention": "randomise/standardise background", "verdict": "shortcut exposed"},
    {"experiment": "Act II", "model": "mitigated CNN", "apparent": "shape task still works", "factor": "shape", "evidence": "swap re-test follows object label", "intervention": "same counterfactual re-test", "verdict": "background shortcut reduced"},
]

fig, axis = plt.subplots(figsize=(15.2, 5.4), constrained_layout=True)
axis.axis("off")
fig.suptitle("Final factor-control table: what controlled the score?", fontsize=18, fontweight="bold")
columns = ["experiment", "model", "apparent result", "controlling factor", "evidence", "intervention", "verdict"]
col_x = [0.02, 0.12, 0.25, 0.42, 0.61, 0.78, 0.91]
for x, column in zip(col_x, columns, strict=True):
    axis.text(x, 0.90, column, fontsize=9.7, fontweight="bold", ha="left", transform=axis.transAxes)
for row, item in enumerate(factor_control_rows):
    y = 0.75 - row * 0.18
    verdict_colour = "#B23A48" if "exposed" in item["verdict"] else MODEL_COLOURS["CNN"]
    axis.add_patch(plt.Rectangle((0.01, y - 0.065), 0.98, 0.12, facecolor="#FAFAFA", edgecolor="#E1E1E1", linewidth=1.0, transform=axis.transAxes))
    values = [item["experiment"], item["model"], item["apparent"], item["factor"], item["evidence"], item["intervention"], item["verdict"]]
    for x, value, column in zip(col_x, values, columns, strict=True):
        colour = verdict_colour if column in {"controlling factor", "verdict"} else "#222222"
        axis.text(x, y, value, fontsize=9.0, color=colour, fontweight="bold" if column in {"controlling factor", "verdict"} else "normal", ha="left", va="center", wrap=True, transform=axis.transAxes)
fig.text(0.5, -0.02, "The useful explanation is not a feature label. It is the factor that controls the model score under intervention.", ha="center", fontsize=12, fontweight="bold")
show_fig(fig)


### Closing

> **The model did not cheat. The data taught the shortcut.**

Accuracy told us the models passed the exam.  
The statistical audits showed the exam contained shortcuts.  
The behavioural probes showed which shortcut the model used.  
The interventions changed the learning problem.  
The re-tests showed whether the learned structure changed.

This is the useful role of XAI: not decoration, but experimental model science.

XAI is not just feature recognition. In this controlled laboratory, explanation means identifying which factor controls the model score, testing whether that factor is legitimate, changing the data-generating process, and re-testing behaviour.

Final thesis: **The model did not cheat. The data did not force the human concept. Many functions solved the exam; XAI reveals which one was learned.**

In [ ]:
assert DATA_MODE == "generated_controlled_demo"
assert EXTERNAL_DATA_REQUIRED is False
assert IMAGE_SIZE == 128

assert mlp_train_accuracy >= 0.995
assert mlp_iid_validation_accuracy >= 0.995
assert mlp_iid_validation_mean_correct_prob >= 0.98
assert mlp_iid_validation_min_correct_prob >= 0.90
assert mlp_ood_accuracy < 0.70
assert mlp_worst_group_accuracy < 0.50
assert mlp_crossed_accuracy <= 0.35
assert mlp_counterfactual_flip_rate > 0.50

assert cnn_iid_validation_accuracy >= 0.995
assert cnn_iid_validation_mean_correct_prob >= 0.98
assert cnn_iid_validation_min_correct_prob >= 0.90
assert cnn_ood_accuracy > 0.85
assert cnn_worst_group_accuracy > mlp_worst_group_accuracy
assert cnn_crossed_accuracy >= 0.90
assert cnn_counterfactual_flip_rate < mlp_counterfactual_flip_rate

assert mlp_cf_predictions == [0, 1, 1, 0]
assert cnn_cf_predictions == [0, 0, 1, 1]
assert selected_mlp_moon_counterfactual_flipped
assert selected_cnn_moon_counterfactual_stable
assert selected_mlp_star_counterfactual_flipped
assert selected_cnn_star_counterfactual_stable
assert selected_cnn_moon_moved_correct_prob >= 0.90
assert selected_cnn_star_moved_correct_prob >= 0.90
assert selected_mlp_moon_moved_wrong_prob >= 0.90
assert selected_mlp_star_moved_wrong_prob >= 0.90

assert "movement_path_results" in globals()
assert "position_response_metrics" in globals()
assert "shape_morph_results" in globals()
assert "controlled_ablation_results" in globals()
assert "minimum_flip_change_results" in globals()
assert "factor_sensitivity_summary" in globals()
assert "many_functions_rows" in globals()
assert "factor_space_diagram_created" in globals()
assert mlp_position_sensitivity > cnn_position_sensitivity
assert cnn_shape_consistency > mlp_shape_consistency

assert many_functions_position_iid_accuracy >= 0.99
assert data_first_position_results["biased train"] >= 0.99
assert data_first_position_results["IID validation"] >= 0.99
assert 0.40 <= data_first_position_results["balanced OOD audit"] <= 0.60
assert nearest_neighbour_shortcut_results["act1_position_iid_accuracy"] >= 0.99
assert nearest_neighbour_shortcut_results["act1_position_uniform_accuracy"] <= 0.70

assert act2_validation_accuracy >= 0.995
assert act2_validation_mean_correct_prob >= 0.98
assert act2_validation_min_correct_prob >= 0.90
assert act2_swap_scores["moon_on_moon_bg"] <= 0.05
assert act2_swap_scores["moon_on_star_bg"] >= 0.95
assert act2_swap_scores["star_on_star_bg"] >= 0.95
assert act2_swap_scores["star_on_moon_bg"] <= 0.05
assert act2_background_shortcut_results["threshold train"] >= 0.99
assert act2_background_shortcut_results["threshold IID validation"] >= 0.99
assert act2_background_shortcut_results["1-NN IID validation"] >= 0.99
assert act2_background_swap_results["moon_on_star_bg"]["threshold_prediction"] == 1
assert act2_background_swap_results["star_on_moon_bg"]["threshold_prediction"] == 0
assert nearest_neighbour_shortcut_results["act2_background_iid_accuracy"] >= 0.99
assert nearest_neighbour_shortcut_results["act2_background_query_prediction"] == 1
assert minimum_flip_change_results["act1_moon_pixels"] is not None
assert minimum_flip_change_results["act2_moon_red_shift_255"] is not None
assert factor_sensitivity_summary["Act I MLP"]["dominant"] == "position"
assert factor_sensitivity_summary["Act II biased CNN"]["dominant"] == "background"
assert factor_sensitivity_summary["Act II mitigated CNN"]["dominant"] == "shape"

assert act2_mitigated_swap_scores["moon_on_moon_bg"] <= 0.10
assert act2_mitigated_swap_scores["moon_on_star_bg"] <= 0.10
assert act2_mitigated_swap_scores["star_on_star_bg"] >= 0.90
assert act2_mitigated_swap_scores["star_on_moon_bg"] >= 0.90

required_animation_embeds = [
    "anim_moon_moves_confidence.gif",
    "anim_star_moves_confidence.gif",
    "anim_response_map_path_mlp.gif",
    "anim_response_map_path_cnn.gif",
    "anim_morph_lower_left.gif",
    "anim_morph_upper_right.gif",
    "anim_invisible_background_morph_moon.gif",
    "anim_invisible_background_morph_star.gif",
    "anim_moon_moves_heatmaps.gif",
    "anim_star_moves_heatmaps.gif",
    "anim_morph_lower_left_heatmaps.gif",
    "anim_morph_upper_right_heatmaps.gif",
]
for filename in required_animation_embeds:
    assert filename in animation_embeds, filename
    assert len(animation_embeds[filename]) > 1024, filename

source_text = NOTEBOOK_PATH.read_text(encoding="utf-8")
for phrase in [
    "blue" + " scene",
    "amber" + " scene",
    "moon on" + " blue",
    "moon on" + " amber",
    "star on" + " blue",
    "star on" + " amber",
    "SCENE" + "_NAMES",
    "scene" + " cue",
]:
    assert phrase not in source_text

# Avoid self-referential source checks by constructing these strings dynamically.
for forbidden in ["FIGURE" + "_DIR", "ANIMATION" + "_DIR", "save" + "_and_show", "fig." + "savefig", "presentation" + "_export_manifest", "<" + "img src=\"../../outputs/"]:
    assert forbidden not in source_text, forbidden

print("Demo 00 self-check passed.")
